# Experiment 1: every reported table, rebuilt from the raw data

This notebook regenerates **all thirteen tables** that Experiment 1 reports in
the thesis, from the frozen probe sets and the model run files, and checks each
one against the numbers actually printed in the thesis.

It is written to be read top to bottom by someone who has never seen the code.
Every statistic used is defined here in full; nothing is imported from the
analysis scripts except two things that must not be restated because they are
the experiment's own definitions:

| Imported | From | Why it is imported and not re-typed |
|---|---|---|
| `legal_options` | `harvest/probe_store.py` | Runs the **deployed validator** to decide which (task, arm) pairs are legal. Re-implementing the rules here would test this notebook's copy of them, not the experiment's. |
| `RUNGS` | `experiments/ex1/prompts.py` | The prompt conditions, exactly as the runs were generated from them. |
| `SWAP_PAIRS`, `CONTROLS`, `APERTURE_FRANKA` | `experiments/ex1/mislabel.py` | Which objects had their names swapped. The authority is the module that performed the swap. |
| `ARM_TYPES` | `core/cell/cell_config.py` | Arm apertures, reach and payload limits. |

Everything else — every population, every rate, every interval — is computed in
the cells below from those inputs.

## How to run it

```bash
cd fourarm
python3 -m jupyter nbconvert --to notebook --execute --inplace \
        notebooks/ex1/ex1_reproduce_tables.ipynb
```

Or open it in Jupyter and run all cells. It uses the standard library only;
`pandas` is used for display if present and is not required. Total runtime is
about thirty seconds, almost all of it reading the 30 run files.

The notebook **fails loudly**. Every section ends in `assert` statements, so a
cell that completes silently is a cell whose numbers agree with the thesis. If
a number in the thesis is edited without re-running the pipeline, the matching
assertion here breaks.

## What the reader gets from each section

Each table section has the same four parts:

1. **What the table reports**, and which research question it serves.
2. **The population** — which states, which trials, and what is excluded.
3. **The computation**, in code, from the loaded rows.
4. **The check** — the generated LaTeX beside the numbers currently in the
   thesis, and an assertion that they agree.

---
# 1. Provenance: where every number comes from

Two kinds of input file, and nothing else:

**Probe sets** (`probes/*.json`) are the frozen decision states. A state is one
snapshot of the cell: which arms are idle, which tasks are queued, where every
object is. They were frozen once, content-hashed, and never regenerated, so
every condition and every model answered the identical set of questions.

**Run files** (`out/*.jsonl`) are the model answers. One line per trial. A trial
is one model, at one prompt condition, on one state, at one repeat. Each line
carries the model's reply, the validator's verdict on it, and a copy of the
state's ground truth, so a run file can be re-scored without the simulator.

The hashes printed below are what pins the two together: every run file records
the `probe_set_hash` it was generated against, and the gate in section 5 refuses
to continue if any of them disagrees with the probe set on disk.

In [1]:
import collections, hashlib, json, math, os, re, statistics, sys, textwrap

# ---------------------------------------------------------------------------
# Locate the fourarm package directory.  It is the one holding out/ and probes/,
# so the notebook works from wherever it is opened.
# ---------------------------------------------------------------------------

def find_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        for cand in (here, os.path.join(here, "fourarm")):
            if (os.path.isdir(os.path.join(cand, "out"))
                    and os.path.isdir(os.path.join(cand, "probes"))):
                return cand
        parent = os.path.dirname(here)
        if parent == here:
            raise RuntimeError("no directory above %s holds both out/ and probes/"
                               % (start or os.getcwd()))
        here = parent

ROOT = find_root()
sys.path.insert(0, ROOT)

# The thesis repository, used only to check the generated tables against what is
# printed.  Set THESIS_REPO in the environment to point somewhere else.  If it is
# absent the notebook still runs and simply reports the checks as skipped.
THESIS_REPO = os.path.expanduser(
    os.environ.get("THESIS_REPO", "~/Desktop/msc-paper"))

print("fourarm root :", ROOT)
print("thesis repo  :", THESIS_REPO,
      "" if os.path.isdir(THESIS_REPO) else "  (NOT FOUND - checks will be skipped)")

fourarm root : /Users/erinsarlak/Downloads/MastersDissertation/fourarm
thesis repo  : /Users/erinsarlak/Desktop/msc-paper 


In [2]:
# ---------------------------------------------------------------------------
# The two frozen probe sets.
# ---------------------------------------------------------------------------
# Cast A is the main object set: ten YCB objects, 162 states.
# Cast B is a disjoint second object set used once, as a generalisation check.

PROBES = {"casta": "probes/ex1_v2.json",
          "castb": "probes/ex1_setb_v1.json"}


def sha256_of(rel):
    h = hashlib.sha256()
    with open(os.path.join(ROOT, rel), "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_probes(cast):
    return json.load(open(os.path.join(ROOT, PROBES[cast])))


PROBE_SET = {cast: load_probes(cast) for cast in PROBES}

for cast, rel in PROBES.items():
    blob = PROBE_SET[cast]
    print("%-6s %-26s %3d states   set hash %s   file sha256 %s"
          % (cast, rel, len(blob["probes"]), blob["hash"][:16], sha256_of(rel)[:16]))

casta  probes/ex1_v2.json         162 states   set hash e23cd23479778f76   file sha256 09ce1e33ecd41d04
castb  probes/ex1_setb_v1.json    108 states   set hash b8abfb3391dd6d52   file sha256 8ade4bab7425cbd4


In [3]:
# ---------------------------------------------------------------------------
# The run files.
# ---------------------------------------------------------------------------
# Named out/ex1_<cast>_<model>_<condition>_r<repeats>.jsonl.  They are listed
# explicitly rather than globbed: out/ also holds smoke tests and repair runs
# that fall into the same (model, condition) bucket, and a glob once pulled one
# of those into a published figure.

MODELS = ["gemini", "gpt", "qwen"]
MODEL_LABEL = {"gemini": "Gemini", "gpt": "GPT", "qwen": "Qwen"}

# Condition key -> (name used in the thesis, thesis tag, rung code in the data).
# The rung code is what prompts.py keys on and what every run row records.
# The tag is the thesis's own shorthand for the condition; it appears in
# Table 4.4 and nowhere in the code, so it is declared here rather than derived.
CONDITIONS = collections.OrderedDict([
    ("full",         ("Full Information",   "P-FULL",           "L3")),
    ("anon",         ("Anonymous",          "P-ANON",           "L3-anon")),
    ("swap",         ("Swapped Names",      "P-SWAP",           "L3-swap")),
    ("nowidth",      ("No Width",           "P-NOWIDTH",        "L3-nowidth")),
    ("nowidth-anon", ("No Width + Anon.",   "P-NOWIDTH-ANON",   "L1-nowidth")),
    ("nowidth-swap", ("No Width + Swapped", "P-NOWIDTH-SWAP",   "L1-swap")),
    ("norules",      ("No Rules",           "P-NORULES",        "L2")),
    ("givenset",     ("Legal-Arm Control",  "P-GIVENSET",       "L4")),
])

CROSSED  = ["full", "anon", "swap", "nowidth", "nowidth-anon", "nowidth-swap"]
CONTROLS = ["norules", "givenset"]
ORDER    = CROSSED + CONTROLS

# Legal-Arm Control was run once; everything else three times.
REPEATS = {cond: (1 if cond == "givenset" else 3) for cond in ORDER}

# Four cells, crossing the width factor with the true and withheld identity
# levels. "anon" was missing here until 2026-09-10, so Table 4.11's
# Anonymous row -- which the thesis prints -- had no generator and the
# audit's "13 of 13 matched" did not cover it.
CASTB_CONDITIONS = ["full", "anon", "nowidth", "nowidth-anon"]

FILES = {}
for model in MODELS:
    for cond in ORDER:
        FILES[("casta", model, cond)] = (
            "out/ex1_casta_%s_%s_r%d.jsonl" % (model, cond, REPEATS[cond]))
for cond in CASTB_CONDITIONS:
    FILES[("castb", "gpt", cond)] = "out/ex1_castb_gpt_%s_r3.jsonl" % cond
    # Cast B was also executed once as a separate single-repeat run, at the same
    # prompt version on the same states.  Reported as a run-to-run stability
    # column and never pooled with the three-repeat run.
    FILES[("castb_r1", "gpt", cond)] = "out/ex1_castb_gpt_%s_r1.jsonl" % cond

# The image-on cell: the one condition where GPT saw a rendered frame as well as
# the text.  Used for a robustness sentence, not for a table.
FILES[("image", "gpt", "nowidth-anon")] = \
    "out/ex1_casta_gpt_nowidth-anon_r3_image.jsonl"

missing = [rel for rel in FILES.values()
           if not os.path.exists(os.path.join(ROOT, rel))]
assert not missing, "run files not found:\n  " + "\n  ".join(missing)
print("%d run files, all present" % len(FILES))

33 run files, all present


In [4]:
# ---------------------------------------------------------------------------
# Load every run file once.  ~7800 rows in total, so this is cheap and it keeps
# every later cell a pure function of ROWS.
# ---------------------------------------------------------------------------

def load_rows(rel):
    with open(os.path.join(ROOT, rel)) as fh:
        return [json.loads(line) for line in fh if line.strip()]

ROWS = {key: load_rows(rel) for key, rel in FILES.items()}

print("%-9s %-7s %-19s %-40s %6s %8s"
      % ("cast", "model", "condition", "file", "rows", "sha256"))
for key in sorted(ROWS):
    cast, model, cond = key
    print("%-9s %-7s %-19s %-40s %6d %8s"
          % (cast, model, CONDITIONS[cond][0], os.path.basename(FILES[key]),
             len(ROWS[key]), sha256_of(FILES[key])[:8]))
print("\ntotal trials loaded:", sum(len(v) for v in ROWS.values()))

cast      model   condition           file                                       rows   sha256
casta     gemini  Anonymous           ex1_casta_gemini_anon_r3.jsonl              486 6f375eaf
casta     gemini  Full Information    ex1_casta_gemini_full_r3.jsonl              486 c17f8349
casta     gemini  Legal-Arm Control   ex1_casta_gemini_givenset_r1.jsonl          162 9d219296
casta     gemini  No Rules            ex1_casta_gemini_norules_r3.jsonl           486 b0ae7ab8
casta     gemini  No Width            ex1_casta_gemini_nowidth_r3.jsonl           486 6bb25f09
casta     gemini  No Width + Anon.    ex1_casta_gemini_nowidth-anon_r3.jsonl      486 961fad16
casta     gemini  No Width + Swapped  ex1_casta_gemini_nowidth-swap_r3.jsonl      486 bd0d5a36
casta     gemini  Swapped Names       ex1_casta_gemini_swap_r3.jsonl              486 aaa9cd81
casta     gpt     Anonymous           ex1_casta_gpt_anon_r3.jsonl                 486 9f734f77
casta     gpt     Full Information    ex1_casta_gp

---
# 2. The states, and the populations built from them

Cast A holds **162 frozen states**. Every reported figure is measured on one of
four subsets of those states, and using the wrong one is the single easiest way
to get an Experiment 1 number wrong. They are defined here once and referred to
by name from then on.

| Population | States | What it is | What is measured on it |
|---|---|---|---|
| All states | 162 | every frozen state | integrity checks only |
| Picking states | 126 | at least one legal (task, arm) pair exists | legality |
| Refusal states | 36 | no legal pair exists at all | correct refusal |
| **Grasp-binding picking states** | **96** | picking states where the grasp constraint blocks at least one pair | **the primary endpoint** |
| Grasp-free picking states | 30 | picking states where grasp blocks nothing | the negative control |

The primary endpoint is legality on the 96 grasp-binding picking states. The
restriction is what makes the measure sensitive to the manipulation: on a state
where grasp blocks nothing, removing the declared grasp width cannot change
which answers are legal, so those states cannot show the effect and are instead
used as the control that proves the effect is specific.

Two traps, both of which have caught this project before and both of which are
asserted against below:

- **The grasp-binding subset is 122 states, but legality is measured on 96.**
  The other 26 grasp-binding states have no legal pair at all, so they are
  scored by correct refusal instead. Expecting 122 x 3 repeats gives 366 trials
  where the real denominator is 288.
- **Scene level means the 96 grasp-binding picking scenes, not all 126 picking
  scenes.** Using all 126 moves Gemini's No Width figure from 71.9 to 78.6.

In [5]:
# ---------------------------------------------------------------------------
# Population predicates.
# ---------------------------------------------------------------------------
# Both probe records and run rows carry the same ground-truth fields, frozen at
# probe-set build time, so the same predicate reads either.  A run row is never
# consulted for its own ground truth: n_legal_pairs and binds_grasp were copied
# in from the probe set and are checked against it in section 5.

def is_picking(rec):
    """True when at least one legal (task, arm) pair exists on this state."""
    if "zero_legal" in rec:                      # run row
        return not rec["zero_legal"]
    return (rec["derived"]["n_legal_pairs"] or 0) > 0   # probe record


def binds_grasp(rec):
    """True when the grasp constraint blocks at least one candidate pair."""
    if "binds_grasp" in rec:                     # run row
        return bool(rec["binds_grasp"])
    return bool((rec["derived"].get("binding_causes") or {}).get("grasp"))


def scene_key(rec):
    """Identifies the state a trial was drawn from.

    A state is identified by where it was harvested from, not by an index into
    the file, so the same state keeps its identity across every condition and
    every repeat.  This is what lets three repeats be collapsed into one verdict.
    """
    p = rec["provenance"]
    return (p["source"], p["seq"], p["round"])


PROBES_A = PROBE_SET["casta"]["probes"]

POPULATIONS = {
    "all":              PROBES_A,
    "picking":          [p for p in PROBES_A if is_picking(p)],
    "refusal":          [p for p in PROBES_A if not is_picking(p)],
    "grasp_binding":    [p for p in PROBES_A if binds_grasp(p)],
    "grasp_picking":    [p for p in PROBES_A if binds_grasp(p) and is_picking(p)],
    "grasp_refusal":    [p for p in PROBES_A if binds_grasp(p) and not is_picking(p)],
    "graspfree_picking":[p for p in PROBES_A if not binds_grasp(p) and is_picking(p)],
}
N = {k: len(v) for k, v in POPULATIONS.items()}

for name in ("all", "picking", "refusal", "grasp_binding", "grasp_picking",
             "grasp_refusal", "graspfree_picking"):
    print("%-20s %3d" % (name, N[name]))

# These seven counts are load-bearing for every table below.
assert N["all"] == 162
assert N["picking"] + N["refusal"] == N["all"] == 162
assert N["picking"] == 126 and N["refusal"] == 36
assert N["grasp_binding"] == 122
assert N["grasp_picking"] == 96 and N["grasp_refusal"] == 26
assert N["grasp_picking"] + N["grasp_refusal"] == N["grasp_binding"]
assert N["graspfree_picking"] == 30
print("\npopulations agree with the chapter")

all                  162
picking              126
refusal               36
grasp_binding        122
grasp_picking         96
grasp_refusal         26
graspfree_picking     30

populations agree with the chapter


---
# 3. Scoring: what counts, and what is excluded

Each trial returns one of three outcomes:

| Outcome | Recorded as | Meaning |
|---|---|---|
| A legal proposal | `valid` | the model named a task and an arm, and the validator accepted the pair |
| An illegal proposal | `rejected` | the model named a task and an arm, and the validator refused it |
| A decline | `noop` | the model replied `task_id -1`, proposing nothing |

**Legality is accepted proposals over proposals made.** A decline is not a
proposal, so it enters neither side of the fraction. This is why the trial
denominators in the tables are not all 288: a model that declines on a picking
state removes that trial from the legality denominator. The rule is stated in
the thesis glossary as *"Declines are not in the denominator"*.

A decline is not thereby forgiven — on a picking state it is still a failure to
allocate, and it is measured directly by the correct-refusal column, which is
the mirror measure: declines over the 36 refusal states, where declining is the
right answer.

Two further exclusions, both rare:

- A row whose reply could not be parsed at all carries `error` and no
  `decision`. There is exactly one such row in cast A (Gemini, No Width +
  Swapped) and it drops out of every rate.
- A scene whose three repeats leave no majority-eligible trial disappears from
  the scene-level denominator. This is why several of GPT's scene denominators
  are 95 rather than 96, and it is the reason the thesis footnotes "95 or 96".

## The unit of analysis

288 trials in a cell are **96 scenes answered three times**, not 288
independent observations. Treating them as independent would make every
interval too narrow. So the three repeats are collapsed into one verdict per
scene by majority — a scene counts as legal when at least two of its three
repeats were legal — and the interval is computed on the 96 scenes.

**Scene-majority is the reported unit throughout Experiment 1.** Trial level is
computed alongside it in this notebook, because a conclusion that changes
between the two would be a finding in itself, but it is not what the tables
print. The exceptions are stated where they occur: raw completeness counts
("0 of 256 reasons"), the Franka share, and the violation counts in Table 4.9
are trial-level quantities and are labelled as such.

In [6]:
# ---------------------------------------------------------------------------
# The measures.  Each returns (successes, denominator) so that the interval
# machinery in section 4 never has to know what was counted.
# ---------------------------------------------------------------------------

PROPOSALS = ("valid", "rejected")     # a decline ("noop") is neither


def legality_trials(rows, grasp=True):
    """Legal proposals over proposals made, on picking states.

    grasp=True  -> the 96 grasp-binding picking states, the primary endpoint.
    grasp=False -> the 30 grasp-free picking states, the negative control.
    """
    k = n = 0
    for r in rows:
        if not is_picking(r) or binds_grasp(r) != grasp:
            continue
        if r.get("result") not in PROPOSALS:
            continue
        n += 1
        k += r["result"] == "valid"
    return k, n


def _by_scene(rows, keep, hit):
    """Group trials by scene, then take a majority of the kept trials."""
    groups = collections.defaultdict(list)
    for r in rows:
        if keep(r):
            groups[scene_key(r)].append(hit(r))
    return sum(1 for v in groups.values() if sum(v) * 2 > len(v)), len(groups)


def legality_scenes(rows, grasp=True):
    """Per-scene majority legality.  THE reported figure."""
    return _by_scene(
        rows,
        lambda r: (is_picking(r) and binds_grasp(r) == grasp
                   and r.get("result") in PROPOSALS),
        lambda r: r["result"] == "valid")


def legality_scenes_all_picking(rows):
    """Per-scene majority legality over all 126 picking states.

    Reported in one column of Table 4.8 only.  Kept separate from
    legality_scenes so the two can never be confused: they differ by seven
    points on Gemini's No Width cell.
    """
    return _by_scene(rows,
                     lambda r: is_picking(r) and r.get("result") in PROPOSALS,
                     lambda r: r["result"] == "valid")


def refusal_scenes(rows):
    """Correct refusal: declines over the refusal states, per-scene majority."""
    return _by_scene(rows, lambda r: not is_picking(r),
                     lambda r: r.get("result") == "noop")


def refusal_trials(rows):
    k = n = 0
    for r in rows:
        if is_picking(r):
            continue
        n += 1
        k += r.get("result") == "noop"
    return k, n


def franka_share(rows):
    """Share of proposals sent to a Franka arm, over every proposal made.

    Trial level and deliberately so: this asks where the answers went, not how
    many scenes were answered correctly.  Two Franka arms and two UR arms, so
    50.0% is what treating the four as interchangeable looks like.
    """
    k = n = 0
    for r in rows:
        arm = (r.get("decision") or {}).get("arm")
        if not arm:
            continue
        n += 1
        k += arm.startswith("franka")
    return k, n


def violations_by_cause(rows):
    """Which constraint each rejected proposal broke.  Trial level."""
    return collections.Counter(r["violation_cause"] for r in rows
                               if r.get("violation_cause"))

ARM_INDIFFERENT = 50.0    # two Franka, two UR
print("measures defined")

measures defined


In [7]:
# ---------------------------------------------------------------------------
# Show the exclusions rather than asserting they are negligible.
# ---------------------------------------------------------------------------
print("Cast A: what happens to the 288 grasp-binding picking trials in each cell\n")
print("%-7s %-19s %7s %7s %8s %8s %7s"
      % ("model", "condition", "trials", "valid", "rejected", "declined", "scored"))
for model in MODELS:
    for cond in ORDER:
        rows = ROWS[("casta", model, cond)]
        sub = [r for r in rows if is_picking(r) and binds_grasp(r)]
        c = collections.Counter(r.get("result") for r in sub)
        k, n = legality_trials(rows)
        print("%-7s %-19s %7d %7d %8d %8d %7d"
              % (MODEL_LABEL[model], CONDITIONS[cond][0], len(sub),
                 c["valid"], c["rejected"], c["noop"], n))

errored = [(os.path.basename(FILES[key]), sum(1 for r in v if r.get("error")))
           for key, v in ROWS.items() if any(r.get("error") for r in v)]
print("\nunparseable replies:", errored or "none")

Cast A: what happens to the 288 grasp-binding picking trials in each cell

model   condition            trials   valid rejected declined  scored
Gemini  Full Information        288     288        0        0     288
Gemini  Anonymous               288     288        0        0     288
Gemini  Swapped Names           288     288        0        0     288
Gemini  No Width                288     208       80        0     288
Gemini  No Width + Anon.        288     217       71        0     288
Gemini  No Width + Swapped      288     218       69        0     287
Gemini  No Rules                288     288        0        0     288
Gemini  Legal-Arm Control        96      96        0        0      96
GPT     Full Information        288     276        7        5     283
GPT     Anonymous               288     276        4        8     280
GPT     Swapped Names           288     268        7       13     275
GPT     No Width                288     206       74        8     280
GPT     No Widt

---
# 4. The statistics

Three quantities, and nothing else, are used anywhere in Experiment 1.

**Wilson score interval** for a single rate. Chosen over the textbook normal
interval because several cells sit at or near 100%, where the normal interval
produces a bound above 100% or a zero-width interval at exactly 100%. Wilson
stays inside [0, 1] and stays sensible at small denominators, which matters when
the refusal population is 36 states.

**Newcombe hybrid-score interval** for the difference of two rates. It is built
from the two Wilson intervals rather than from a pooled normal approximation, so
it inherits the same good behaviour near the boundaries. Every contrast reported
in the chapter — every "+28.1 [19.2, 37.8]" — is a Newcombe interval on two
scene-majority rates.

**Difference of two differences**, by normal approximation, for the interaction
test and for the swap separation. Used only where both inputs sit well away from
0 and 100, which is where the approximation is adequate.

All three are given at 95%, two-sided, z = 1.96.

In [8]:
def wilson(k, n, z=1.96):
    """95% Wilson score interval for k successes in n trials, as percentages."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (100 * max(0.0, centre - half), 100 * min(1.0, centre + half))


def newcombe(k1, n1, k2, n2, z=1.96):
    """Group 1 minus group 2, Newcombe hybrid-score interval, as percentages.

    Returns (difference, lower, upper).  Built from the two Wilson intervals:
    the lower bound uses group 1's lower and group 2's upper, and vice versa.
    """
    l1, u1 = (x / 100 for x in wilson(k1, n1, z))
    l2, u2 = (x / 100 for x in wilson(k2, n2, z))
    p1, p2 = k1 / n1, k2 / n2
    d = p1 - p2
    return (100 * d,
            100 * (d - math.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2)),
            100 * (d + math.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)))


def diff_of_diffs(a, b):
    """Difference between two Newcombe differences, normal approximation.

    Each input is a (difference, lower, upper) triple.  The standard error is
    recovered from the interval width, 3.92 being 2 x 1.96.
    """
    se = math.sqrt(((a[2] - a[1]) / 3.92) ** 2 + ((b[2] - b[1]) / 3.92) ** 2)
    d = a[0] - b[0]
    return d, d - 1.96 * se, d + 1.96 * se


def pct(k, n):
    return float("nan") if n == 0 else 100.0 * k / n


def fmt_rate(k, n):
    """'71.9 [62.2, 79.9]' - a rate with its Wilson interval."""
    lo, hi = wilson(k, n)
    return "%.1f [%.1f, %.1f]" % (pct(k, n), lo, hi)


def fmt_diff(base, manip):
    """'+28.1 [+19.2, +37.8]' - base minus manipulated, Newcombe interval."""
    d, lo, hi = newcombe(base[0], base[1], manip[0], manip[1])
    return "%+.1f [%+.1f, %+.1f]" % (d, lo, hi)


def spans_zero(lo, hi):
    return lo <= 0 <= hi


# Sanity checks on the interval code itself, against values that can be
# verified by hand or against a published worked example.
assert wilson(0, 36) == (0.0, wilson(0, 36)[1])            # lower bound pinned at 0
assert round(wilson(0, 36)[1], 1) == 9.6                   # rule-of-three style bound
assert round(wilson(96, 96)[0], 1) == 96.2                 # 100% does not give a point interval
lo, hi = wilson(69, 96)
assert (round(pct(69, 96), 1), round(lo, 1), round(hi, 1)) == (71.9, 62.2, 79.9)
d, dlo, dhi = newcombe(96, 96, 69, 96)
assert (round(d, 1), round(dlo, 1), round(dhi, 1)) == (28.1, 19.2, 37.8)
print("interval functions agree with the figures printed in the chapter")

interval functions agree with the figures printed in the chapter


In [9]:
# ---------------------------------------------------------------------------
# Display and LaTeX.  Tables are built as (headers, rows) and rendered twice:
# once for the screen, once as the LaTeX the thesis inputs.
# ---------------------------------------------------------------------------

try:
    import pandas as pd
    def show(headers, rows, title=None):
        if title:
            print(title)
        display(pd.DataFrame(rows, columns=headers))
except ImportError:
    pd = None
    def show(headers, rows, title=None):
        if title:
            print(title)
        cols = [max(len(str(h)), *(len(str(r[i])) for r in rows)) if rows
                else len(str(h)) for i, h in enumerate(headers)]
        line = "  ".join("-" * c for c in cols)
        print(line)
        print("  ".join(str(h).ljust(c) for h, c in zip(headers, cols)))
        print(line)
        for r in rows:
            print("  ".join(str(x).ljust(c) for x, c in zip(r, cols)))
        print(line)


TEX_OUT = os.path.join(ROOT, "tables", "ex1")
os.makedirs(TEX_OUT, exist_ok=True)

# Written tables are registered here so section 20 can audit them.
BUILT = collections.OrderedDict()


def tex_escape(s):
    """Escape a header or cell for LaTeX, leaving anything already escaped alone.

    Headers arrive part-escaped - "Change [95\\% CI]" is written that way at the
    call site - so a blanket replace would double the backslash.  Only an
    unescaped underscore or ampersand is touched; a bare arm_state header
    otherwise reaches LaTeX as a subscript and the file does not compile.
    """
    s = re.sub(r"(?<!\\)_", r"\\_", str(s))
    return re.sub(r"(?<!\\)&(?!\s)", r"\\&", s)


def blank_repeats(rows, column=0):
    """Blank a repeated value in the leading column, as the thesis prints it."""
    out, previous = [], None
    for r in rows:
        r = list(r)
        if r[column] == previous:
            r[column] = ""
        else:
            previous = r[column]
        out.append(r)
    return out


def latex_table(label, headers, rows, caption, colspec=None, note=None,
                size="small"):
    """Render a table in the form the thesis inputs, and write it to tables/ex1/.

    The filename is derived from the label - tab:ex1:design becomes
    ex1_design.tex - so a table and its \\label can never drift apart.
    """
    colspec = colspec or "l" * len(headers)
    body = ["\\begin{table}[H]", "  \\centering",
            "  \\caption{%s}" % caption, "  \\label{%s}" % label,
            "  \\%s" % size,
            "  \\begin{tabular}{%s}" % colspec, "    \\toprule",
            "    " + " & ".join(tex_escape(h) for h in headers) + r" \\",
            "    \\midrule"]
    body += ["    " + " & ".join(str(c) for c in r) + r" \\" for r in rows]
    body += ["    \\bottomrule", "  \\end{tabular}"]
    if note:
        body.append("  \\begin{minipage}{\\linewidth}\\vspace{2pt}\\footnotesize "
                    + note + "\\end{minipage}")
    body.append("\\end{table}")
    tex = "\n".join(body) + "\n"

    stem = label.replace("tab:", "").replace(":", "_")
    path = os.path.join(TEX_OUT, stem + ".tex")
    with open(path, "w") as fh:
        fh.write(tex)
    BUILT[label] = {"path": path, "headers": headers, "rows": rows}
    return tex

In [10]:
# ---------------------------------------------------------------------------
# Checking a generated table against the thesis.
# ---------------------------------------------------------------------------
# Six Experiment 1 tables live in their own files under thesis/tables/; the
# other seven are written inline in main.tex.  Both are read the same way: pull
# out the table environment carrying the label, then take the numbers from the
# rows between \midrule and \bottomrule.
#
# The comparison is on the ordered sequence of numeric tokens rather than on the
# text, so it is immune to spacing and to caption edits and sensitive to exactly
# what it should be sensitive to: a changed number.

def thesis_source(label):
    """The LaTeX of one thesis table, from its own file or from main.tex."""
    if not os.path.isdir(THESIS_REPO):
        return None
    stem = label.replace("tab:", "").replace(":", "_")
    own = os.path.join(THESIS_REPO, "tables", stem + ".tex")
    if os.path.exists(own):
        return open(own).read()
    main = open(os.path.join(THESIS_REPO, "main.tex")).read()
    marker = "\\label{%s}" % label
    at = main.find(marker)
    if at < 0:
        return None
    start = main.rfind("\\begin{table}", 0, at)
    end = main.find("\\end{table}", at)
    return main[start:end + len("\\end{table}")]


NUMBER = re.compile(r"-?\d+\.?\d*")


def table_body(tex):
    """The row block of a LaTeX table: everything between the FIRST \\midrule
    and \\bottomrule.

    Splitting on the last \\midrule instead would silently reduce a table with
    an internal rule - binding, subsets, rungs all have one - to its final block,
    and a check that compares three rows out of nine is not a check.
    """
    after = tex.split("\\midrule", 1)[1]
    return after.split("\\bottomrule")[0].replace("\\midrule", " ")


def table_numbers(tex):
    """Every number in the body rows of a LaTeX table, in order.

    Only the rows of the table are read, so numbers appearing in a caption or a
    note are ignored: those are prose about the table, not cells of it.
    """
    body = table_body(tex)
    out = []
    for line in body.splitlines():
        line = re.sub(r"\\(num|SI|si|texttt|textbf|emph|quad|multicolumn)", " ", line)
        line = re.sub(r"table-format=[\d.]+", " ", line)
        out += [float(x) for x in NUMBER.findall(line)]
    return out


CHECKS = []      # (label, status, detail) - collected for the audit in section 20


def check_against_thesis(label, tex, note=""):
    """Assert that a generated table carries the same numbers as the thesis."""
    published = thesis_source(label)
    if published is None:
        CHECKS.append((label, "SKIPPED", "thesis source not found"))
        print("%-22s SKIPPED  (thesis source not found)" % label)
        return
    ours, theirs = table_numbers(tex), table_numbers(published)
    if ours == theirs:
        CHECKS.append((label, "MATCHES", "%d numeric cells%s"
                       % (len(ours), (" - " + note) if note else "")))
        print("%-22s MATCHES the thesis on all %d numeric cells" % (label, len(ours)))
        return
    diffs = [(i, a, b) for i, (a, b) in enumerate(zip(ours, theirs)) if a != b]
    detail = ("computed %d numbers, thesis has %d" % (len(ours), len(theirs))
              if len(ours) != len(theirs)
              else "; ".join("pos %d: computed %s, thesis %s" % d for d in diffs[:8]))
    CHECKS.append((label, "DIFFERS", detail))
    raise AssertionError("%s does not match the thesis: %s" % (label, detail))

---
# 5. The integrity gate

Nothing below this cell is trustworthy if this cell does not pass. It checks the
things that, when wrong, produce a plausible-looking number rather than a crash:

1. **Row counts.** 486 rows is 162 states x 3 repeats; 162 is a single repeat.
   A short file means a run was interrupted and the missing states are not
   random — they are whichever states came last.
2. **Repeat balance.** Every state answered the same number of times. An
   unbalanced file silently weights some states more than others.
3. **The right prompt condition.** Every row records the rung code it was
   generated from. A file that answered a different condition than its name
   claims is the failure mode that a naming scheme cannot catch.
4. **The right probe set.** Every row records the hash of the probe set it was
   drawn from, which must equal the hash of the probe set on disk.
5. **Ground truth agrees with the probe set.** The legality of a state is
   frozen in the probe set; each run row carries a copy. If a copy disagrees,
   the state was re-derived at some point and the run is measuring something
   else.

In [11]:
GATE = []

def gate(name, got, want):
    ok = got == want
    GATE.append((name, got, want, ok))
    return ok


truth = {scene_key(p): (is_picking(p), binds_grasp(p)) for p in PROBES_A}
probe_hash_a = PROBE_SET["casta"]["hash"]

for model in MODELS:
    for cond in ORDER:
        key, rows = ("casta", model, cond), ROWS[("casta", model, cond)]
        tag = "%s/%s" % (model, cond)
        reps = REPEATS[cond]

        gate(tag + " rows", len(rows), 162 * reps)

        per_scene = collections.Counter(scene_key(r) for r in rows)
        gate(tag + " states covered", len(per_scene), 162)
        gate(tag + " repeats balanced", set(per_scene.values()), {reps})

        gate(tag + " rung", {r.get("rung") for r in rows},
             {CONDITIONS[cond][2]})
        gate(tag + " probe set", {r.get("probe_set_hash") for r in rows},
             {probe_hash_a})

        mismatched = sum(1 for r in rows
                         if truth[scene_key(r)] != (is_picking(r), binds_grasp(r)))
        gate(tag + " ground truth matches probe set", mismatched, 0)

# Cast B, same checks against its own probe set.
probe_hash_b = PROBE_SET["castb"]["hash"]
for cond in CASTB_CONDITIONS:
    for cast, reps in (("castb", 3), ("castb_r1", 1)):
        rows = ROWS[(cast, "gpt", cond)]
        tag = "%s/gpt/%s" % (cast, cond)
        gate(tag + " rows", len(rows), 108 * reps)
        gate(tag + " repeats balanced",
             set(collections.Counter(scene_key(r) for r in rows).values()), {reps})
        gate(tag + " rung", {r.get("rung") for r in rows}, {CONDITIONS[cond][2]})
        gate(tag + " probe set", {r.get("probe_set_hash") for r in rows}, {probe_hash_b})

failed = [g for g in GATE if not g[3]]
for name, got, want, ok in failed:
    print("FAIL  %-50s got %r, expected %r" % (name, got, want))
print("integrity gate: %d of %d checks passed" % (len(GATE) - len(failed), len(GATE)))
assert not failed, "%d integrity checks failed - stop here" % len(failed)

integrity gate: 176 of 176 checks passed


---
# 6. Table 4.1 — one frozen state, worked through

**Reports:** a single decision state in full, so the reader can see why legality
cannot be read off any one field.

**Source:** `probes/ex1_v2.json`, the state harvested from `rec_decision_rich`
at sequence 13. No model is involved: every cell is the deployed validator's
verdict on a hypothetical (task, arm) pair.

**Why this state:** it is the smallest state that exercises three different
constraints at once — delicacy, reach and grasp — and it has an arm that is
busy, so the reader sees why the arm count is three and not four.

The legality verdicts come from `legal_options`, which builds a validator from
the frozen state and runs the real `validate_decision` on every candidate pair.
The reason attached to a rejected pair is the first check that failed, because
the validator is a chain of early returns and stops at the first one.

In [12]:
from harvest.probe_store import legal_options
import core.cell.cell_config as CELL
from ycb.ycb_objects import YCB          # the object catalogue: mass, width, delicacy

WORKED = ("rec_decision_rich", 13)

state = next(p for p in PROBES_A
             if (p["provenance"]["source"], p["provenance"]["seq"]) == WORKED)

pairs, per_task, causes = legal_options(state, with_causes=True)
legal_set = {(t, a) for t, a, _kind in pairs}
cause_of = {(t, a): c for t, a, c in causes}

idle = [a["name"] for a in state["state"]["arms"]
        if a["state"] == "IDLE" and not a.get("disabled")]
busy = [(a["name"], a.get("holding")) for a in state["state"]["arms"]
        if a["name"] not in idle]
open_tasks = [t for t in state["state"]["tasks"] if t["status"] == "queued"]

print("state          %s seq %d" % WORKED)
print("idle arms      %s" % ", ".join(idle))
print("busy arms      %s" % ", ".join("%s (holding %s)" % b for b in busy))
print("open tasks     %s" % ", ".join(str(t["id"]) for t in open_tasks))
print("candidate pairs %d x %d = %d" % (len(open_tasks), len(idle),
                                        len(open_tasks) * len(idle)))
print("legal pairs    %d  %s" % (len(legal_set), sorted(legal_set)))

state          rec_decision_rich seq 13
idle arms      ur_w, ur_e, franka_s
busy arms      franka_n (holding ycb_banana)
open tasks     7, 9, 10
candidate pairs 3 x 3 = 9
legal pairs    2  [(9, 'ur_w'), (10, 'ur_w')]


In [13]:
# Object properties come from the catalogue the cell was configured from, not
# from the state record: the state stores placement, the catalogue stores the
# physical fields.  States name objects with a "ycb_" prefix and the catalogue
# is keyed without it, so the prefix is stripped on lookup.
def spec_of(name):
    """The catalogue entry for an object named as the states name it."""
    key = name[4:] if name.startswith("ycb_") else name
    assert key in YCB, "%s is not in the object catalogue" % name
    return YCB[key]


def obj_row(name):
    spec = spec_of(name)
    return spec["grasp_m"], bool(spec.get("delicate"))

CAUSE_WORD = {"delicate": "delicacy", "reach": "reach", "grasp": "grasp",
              "payload": "payload", "no_route": "route"}

rows = []
for t in open_tasks:
    width, delicate = obj_row(t["object"])
    pretty = t["object"].replace("ycb_", "").replace("_", " ")
    src_zone = next(o["zone"] for o in state["state"]["objects"]
                    if o["name"] == t["object"])
    cells = []
    for arm in idle:
        if (t["id"], arm) in legal_set:
            cells.append(r"\textbf{legal}")
        else:
            cells.append("no (%s)" % CAUSE_WORD[cause_of[(t["id"], arm)]])
    rows.append([t["id"], pretty, "%.3f" % width, "yes" if delicate else "no",
                 r"\texttt{%s}" % src_zone, r"\texttt{%s}" % t["dest_zone"]] + cells)

show(["ID", "Deliver", "Width (m)", "Delicate", "From", "To"] + idle, rows,
     "Table 4.1, rebuilt")

Table 4.1, rebuilt
--  -----------  ---------  --------  -----------  -----------  --------------  -------------  ----------
ID  Deliver      Width (m)  Delicate  From         To           ur_w            ur_e           franka_s  
--  -----------  ---------  --------  -----------  -----------  --------------  -------------  ----------
7   bowl         0.030      yes       \texttt{nw}  \texttt{ne}  no (delicacy)   no (delicacy)  no (reach)
9   large clamp  0.122      no        \texttt{sw}  \texttt{sw}  \textbf{legal}  no (reach)     no (grasp)
10  wood block   0.090      no        \texttt{sw}  \texttt{sw}  \textbf{legal}  no (reach)     no (grasp)
--  -----------  ---------  --------  -----------  -----------  --------------  -------------  ----------


In [14]:
# This table carries a two-level header and a threeparttable note, so it is
# emitted in the thesis's own shape rather than through latex_table().
arm_cols = " & ".join(r"\texttt{%s}" % a.replace("_", r"\_") for a in idle)
tex_worked = "\n".join([
    r"\begin{table}[H]", r"  \centering", r"  \begin{threeparttable}",
    r"    \caption{One frozen state, \texttt{%s} seq \num{%d}. %d"
    % (WORKED[0].replace("_", r"\_"), WORKED[1], len(open_tasks)),
    r"      open tasks and %d idle arms give %s candidate pairs. Only %s of them are legal.}"
    % (len(idle), len(open_tasks) * len(idle), {1: "one", 2: "two", 3: "three"}.get(len(legal_set), len(legal_set))),
    r"    \label{tab:ex1:worked}", r"    \footnotesize",
    r"    \setlength{\tabcolsep}{4pt}",
    r"    \begin{tabular}{ll%s}" % ("c" * (4 + len(idle))),
    r"      \toprule",
    r"      \multicolumn{6}{c}{The task} & \multicolumn{%d}{c}{Is this arm legal for it?} \\" % len(idle),
    r"      \cmidrule(lr){1-6}\cmidrule(lr){7-%d}" % (6 + len(idle)),
    r"      ID & Deliver & Width (\si{\metre}) & Delicate & From & To",
    r"        & " + arm_cols + r" \\",
    r"      \midrule",
] + ["      " + " & ".join(str(c) for c in r) + r" \\" for r in rows] + [
    r"      \bottomrule", r"    \end{tabular}",
    r"    \begin{tablenotes}[flushleft]\footnotesize",
    r"      \item The second",
    r"        Franka, \texttt{%s}, is holding the %s and is not offered."
    % (busy[0][0].replace("_", r"\_"), busy[0][1].replace("ycb_", "").replace("_", " ")),
    r"    \end{tablenotes}", r"  \end{threeparttable}", r"\end{table}", ""])

open(os.path.join(TEX_OUT, "ex1_worked.tex"), "w").write(tex_worked)
BUILT["tab:ex1:worked"] = {"path": os.path.join(TEX_OUT, "ex1_worked.tex"),
                           "headers": None, "rows": rows}
check_against_thesis("tab:ex1:worked", tex_worked)

tab:ex1:worked         MATCHES the thesis on all 6 numeric cells


---
# 7. Table 4.2 — which constraints bind, and how often

**Reports:** how many states each constraint blocks something in, and how many
candidate pairs it accounts for.

**Source:** `probes/ex1_v2.json`, the `binding_causes` block computed for each
state when the probe set was frozen. No model is involved.

**What is being counted.** A candidate pair is one open task offered to one idle
arm. Across the 162 states there are 1732 of them; the validator accepts 536, so
1196 are rejected, and this table splits those 1196 by cause. The word
"rejected" also describes a *model* proposal the validator turned down, which is
a far smaller number — the two must not be read together.

**One cause per rejected pair.** A pair can fail several checks at once, but the
validator is a chain of early returns, so it reports the first failure and stops:
capability (delicacy, then grasp width, then payload), then reach, then routing.
That makes the `reach` and `no_route` rows **floors** rather than exact counts,
because a pair that would also have failed reach is recorded under grasp if
grasp was tested first. The delicacy row is exact because nothing precedes it,
and the grasp row is exact *on this cast* because no object in it is both
delicate and wider than an aperture.

The two structural checks at the end are what catch a dropped or
double-counted cause: the pairs column must total the 1196 rejected pairs, and
the states column must *not* total 162, because several constraints can bind on
the same state.

In [15]:
CAUSE_ORDER = ["reach", "grasp", "delicate", "no_route", "payload"]
CAUSE_LABEL = {"reach": "Reach", "grasp": "Grasp", "delicate": "Delicacy",
               "no_route": "Route", "payload": "Payload"}

pairs_by_cause, states_by_cause = collections.Counter(), collections.Counter()
for p in PROBES_A:
    for cause, n in (p["derived"].get("binding_causes") or {}).items():
        if n:
            pairs_by_cause[cause] += n      # every rejected pair it accounts for
            states_by_cause[cause] += 1     # the state counts once, however many pairs

n_candidate = sum(p["derived"]["n_idle_arms"] * p["derived"]["n_tasks_open"]
                  for p in PROBES_A)
n_legal_pairs = sum(p["derived"]["n_legal_pairs"] for p in PROBES_A)
n_rejected = n_candidate - n_legal_pairs

rows = [[CAUSE_LABEL[c], states_by_cause[c], pairs_by_cause[c],
         "%.1f" % (pairs_by_cause[c] / states_by_cause[c])
         if states_by_cause[c] else "--"]
        for c in CAUSE_ORDER]
show(["Constraint", "States", "Pairs", "Pairs per binding state"], rows,
     "Table 4.2, rebuilt")

print("\ncandidate pairs %4d = sum over states of (idle arms x open tasks)" % n_candidate)
print("legal pairs     %4d" % n_legal_pairs)
print("rejected pairs  %4d" % n_rejected)

# Structural checks.
assert sum(pairs_by_cause.values()) == n_rejected == 1196, \
    "the pairs column must account for every rejected pair"
assert states_by_cause["payload"] == 0, \
    "payload is published as a tested-and-inert zero, not omitted"
assert sum(states_by_cause.values()) > len(PROBES_A), \
    "the states column overlaps by design; a partition would mean a lost cause"
print("pairs column totals the rejected pairs; states column overlaps as expected")

Table 4.2, rebuilt
----------  ------  -----  -----------------------
Constraint  States  Pairs  Pairs per binding state
----------  ------  -----  -----------------------
Reach       146     516    3.5                    
Grasp       122     493    4.0                    
Delicacy    80      135    1.7                    
Route       44      52     1.2                    
Payload     0       0      --                     
----------  ------  -----  -----------------------

candidate pairs 1732 = sum over states of (idle arms x open tasks)
legal pairs      536
rejected pairs  1196
pairs column totals the rejected pairs; states column overlaps as expected


In [16]:
# The published table puts payload below a rule, as an explicit zero.
body = [r for r in rows if r[0] != "Payload"] + [[r"\midrule"]] \
       + [r for r in rows if r[0] == "Payload"]
tex = "\n".join([
    r"\begin{table}[H]", r"\centering",
    r"\caption{Binding constraints across the \num{162} frozen states, by "
    r"states affected and candidate pairs blocked.}",
    r"\label{tab:ex1:binding}",
    r"\begin{tabular}{lrrr}", r"\toprule",
    r"Constraint & States & Pairs & Pairs per binding state \\", r"\midrule"]
    + ["%s \\\\" % " & ".join(str(c) for c in r) if len(r) > 1 else r"\midrule"
       for r in body]
    + [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""])
open(os.path.join(TEX_OUT, "ex1_binding.tex"), "w").write(tex)
BUILT["tab:ex1:binding"] = {"path": os.path.join(TEX_OUT, "ex1_binding.tex"),
                            "headers": None, "rows": rows}
check_against_thesis("tab:ex1:binding", tex)

tab:ex1:binding        MATCHES the thesis on all 14 numeric cells


---
# 8. Table 4.3 — the states split two ways

**Reports:** the 162 states crossed by whether grasp binds and whether a legal
assignment exists. This is the table that defines every denominator in the
results chapter.

**Source:** `probes/ex1_v2.json`, via the population predicates from section 2.

The 96 in the top-left cell is the primary subset. The 30 below it is the
negative control: grasp blocks nothing there, so removing the declared grasp
width cannot change which answers are legal, and any movement in that cell is
evidence the manipulation disturbed something other than the grasp comparison.

In [17]:
rows = [["Grasp blocks something", N["grasp_picking"], N["grasp_refusal"],
         N["grasp_binding"]],
        ["Grasp blocks nothing", N["graspfree_picking"],
         N["refusal"] - N["grasp_refusal"],
         N["all"] - N["grasp_binding"]],
        ["Total", N["picking"], N["refusal"], N["all"]]]
show(["", "Picking", "Refusal", "Total"], rows, "Table 4.3, rebuilt")

# The cross-tabulation must close in both directions.
assert rows[0][1] + rows[1][1] == rows[2][1] == 126
assert rows[0][2] + rows[1][2] == rows[2][2] == 36
assert rows[0][3] + rows[1][3] == rows[2][3] == 162
assert all(r[1] + r[2] == r[3] for r in rows)

tex = "\n".join([
    r"\begin{table}[H]", r"\centering",
    r"\caption{Probe states split by whether grasp binds and whether a legal",
    r"assignment exists.}", r"\label{tab:ex1:subsets}",
    r"\begin{tabular}{lrrr}", r"\toprule",
    r"& Picking & Refusal & Total \\",
    r"& \emph{(a legal pair exists)} & \emph{(none exists)} & \\", r"\midrule",
    r"Grasp blocks something   &  %d & %d & %d \\" % tuple(rows[0][1:]),
    r"Grasp blocks nothing     &  %d & %d & %d \\" % tuple(rows[1][1:]),
    r"\midrule",
    r"Total                    & %d & %d & %d \\" % tuple(rows[2][1:]),
    r"\bottomrule", r"\end{tabular}", r"\end{table}", ""])
open(os.path.join(TEX_OUT, "ex1_subsets.tex"), "w").write(tex)
BUILT["tab:ex1:subsets"] = {"path": os.path.join(TEX_OUT, "ex1_subsets.tex"),
                            "headers": None, "rows": rows}
check_against_thesis("tab:ex1:subsets", tex)

Table 4.3, rebuilt
----------------------  -------  -------  -----
                        Picking  Refusal  Total
----------------------  -------  -------  -----
Grasp blocks something  96       26       122  
Grasp blocks nothing    30       10       40   
Total                   126      36       162  
----------------------  -------  -------  -----


tab:ex1:subsets        MATCHES the thesis on all 9 numeric cells


---
# 9. Tables 4.4 and 4.5 — the prompt conditions

**Report:** what information each of the eight prompt conditions supplies
(4.4), and the 2 x 3 crossing of the six that form the design (4.5).

**Source:** `RUNGS` in `experiments/ex1/prompts.py` — the dictionary the runs
were actually generated from. These two tables are design documentation rather
than results, but they are generated here anyway, because a condition table that
disagrees with the code that built the prompts is worse than no condition table:
every contrast in the chapter is named by it.

The mapping from a rung's flags to the identity column is the one substantive
step:

- `names: True`, no `mislabel` — the true object name is shown.
- `names: False` — the name is withheld and the object is described by its
  physical fields alone.
- `mislabel: True` — a name *is* shown but it is the wrong one, swapped across
  the Franka aperture with a category-matched partner. `names` stays `True`
  because a name is supplied; anonymising would erase the very swap being tested.

In [18]:
from experiments.ex1.prompts import RUNGS

def identity_of(flags):
    if flags.get("mislabel"):
        return "swapped"       # a name is shown, and it is the wrong one
    return "true" if flags["names"] else "withheld"

CHECK, DASH = r"\checkmark", "--"
rows, roster = [], []
for cond in ORDER:
    name, tag, rung = CONDITIONS[cond]
    flags = RUNGS[rung]
    ident = identity_of(flags)
    roster.append((cond, name, tag, rung, flags["declared_width"], ident,
                   flags["rules"], flags["eligible"]))
    rows.append([name, r"\texttt{%s}" % tag,
                 CHECK if flags["declared_width"] else DASH,
                 {"true": "true", "withheld": "withheld", "swapped": "false"}[ident],
                 CHECK if flags["rules"] else DASH,
                 CHECK if flags["eligible"] else DASH])

show(["Condition", "Tag", "Width", "Identity", "Capability rules", "Legal set"],
     rows, "Table 4.4, rebuilt from RUNGS")

# The design is what the chapter claims it is: six crossed cells covering every
# combination of two width levels and three identity levels, plus two controls
# that each change exactly one further thing.
crossed = [r for r in roster if r[0] in CROSSED]
assert {(r[4], r[5]) for r in crossed} == \
       {(w, i) for w in (True, False) for i in ("true", "withheld", "swapped")}, \
       "the six crossed conditions must fill the 2 x 3 grid exactly once each"
assert all(r[6] and not r[7] for r in crossed), \
       "every crossed condition keeps the rules and withholds the legal set"
norules = next(r for r in roster if r[0] == "norules")
givenset = next(r for r in roster if r[0] == "givenset")
assert not norules[6] and norules[4] and norules[5] == "true", \
       "No Rules removes the rules and changes nothing else"
assert givenset[7] and givenset[6] and givenset[4], \
       "Legal-Arm Control adds the legal set and removes nothing"
print("\nthe condition roster matches the design described in the chapter")

Table 4.4, rebuilt from RUNGS
------------------  -----------------------  ----------  --------  ----------------  ----------
Condition           Tag                      Width       Identity  Capability rules  Legal set 
------------------  -----------------------  ----------  --------  ----------------  ----------
Full Information    \texttt{P-FULL}          \checkmark  true      \checkmark        --        
Anonymous           \texttt{P-ANON}          \checkmark  withheld  \checkmark        --        
Swapped Names       \texttt{P-SWAP}          \checkmark  false     \checkmark        --        
No Width            \texttt{P-NOWIDTH}       --          true      \checkmark        --        
No Width + Anon.    \texttt{P-NOWIDTH-ANON}  --          withheld  \checkmark        --        
No Width + Swapped  \texttt{P-NOWIDTH-SWAP}  --          false     \checkmark        --        
No Rules            \texttt{P-NORULES}       \checkmark  true      --                --        
Legal-Arm 

In [19]:
tex_rungs = "\n".join([
    r"\begin{table}[H]", r"\centering",
    r"\caption{The eight prompt conditions and the information each supplies.}",
    r"\label{tab:ex1:rungs}", r"\small",
    r"\setlength{\tabcolsep}{4.5pt}",
    r"\begin{tabular}{llccccc}", r"\toprule",
    r"Condition & Tag & Width & Identity & \shortstack{Capability rules} &",
    r"\shortstack{Legal set} \\", r"\midrule"]
    + ["%s \\\\" % " & ".join(str(c) for c in r) for r in rows[:6]]
    + [r"\midrule"]
    + ["%s \\\\" % " & ".join(str(c) for c in r) for r in rows[6:]]
    + [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""])
open(os.path.join(TEX_OUT, "ex1_rungs.tex"), "w").write(tex_rungs)
BUILT["tab:ex1:rungs"] = {"path": os.path.join(TEX_OUT, "ex1_rungs.tex"),
                          "headers": None, "rows": rows}

# Table 4.5 is the same six crossed conditions laid out as the grid they form.
cell = {(r[4], r[5]): r[1] for r in crossed}
grid_rows = [["Present"] + [cell[(True, i)] for i in ("true", "withheld", "swapped")],
             ["Absent"] + [cell[(False, i)] for i in ("true", "withheld", "swapped")]]
show(["Declared width", "True", "Withheld", "False (swapped)"], grid_rows,
     "Table 4.5, rebuilt")

tex_grid = "\n".join([
    r"\begin{table}[H]", r"\centering",
    r"\caption{The $2\times3$ crossed design.}", r"\label{tab:ex1:grid}",
    r"\renewcommand{\arraystretch}{1.4}",
    r"\begin{tabular}{lccc}", r"\toprule",
    r"& \multicolumn{3}{c}{Object identity} \\", r"\cmidrule(l){2-4}",
    r"Declared width & True & Withheld & False (swapped) \\", r"\midrule"]
    + ["%s \\\\" % " & ".join(r) for r in grid_rows]
    + [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""])
open(os.path.join(TEX_OUT, "ex1_grid.tex"), "w").write(tex_grid)
BUILT["tab:ex1:grid"] = {"path": os.path.join(TEX_OUT, "ex1_grid.tex"),
                         "headers": None, "rows": grid_rows}

# Neither table carries a number, so the numeric check has nothing to compare.
# What is checked instead is that the condition NAMES printed in the thesis are
# the ones this roster produces.
for label, expect in (("tab:ex1:rungs", [r[0] for r in rows]),
                      ("tab:ex1:grid", [c for r in grid_rows for c in r[1:]])):
    published = thesis_source(label)
    if published is None:
        CHECKS.append((label, "SKIPPED", "thesis source not found"))
        print("%-22s SKIPPED" % label); continue
    body = table_body(published)
    missing = [name for name in expect
               if name.replace(" + ", " $+$ ") not in body and name not in body]
    assert not missing, "%s: condition names missing from the thesis: %s" % (label, missing)
    CHECKS.append((label, "MATCHES", "condition names, no numeric cells"))
    print("%-22s MATCHES the thesis on all %d condition names" % (label, len(expect)))

Table 4.5, rebuilt
--------------  ----------------  ----------------  ------------------
Declared width  True              Withheld          False (swapped)   
--------------  ----------------  ----------------  ------------------
Present         Full Information  Anonymous         Swapped Names     
Absent          No Width          No Width + Anon.  No Width + Swapped
--------------  ----------------  ----------------  ------------------
tab:ex1:rungs          MATCHES the thesis on all 8 condition names
tab:ex1:grid           MATCHES the thesis on all 6 condition names


---
# 10. Table 4.6 — the two reference lines

**Reports:** the chance floor and the width-blind line, per population. These
are what every legality figure in the chapter is read against.

**Source:** recomputed here from `probes/ex1_v2.json` by re-running the deployed
validator. The pipeline caches these in `out/ex1_chance_floor.json`; this
notebook recomputes them from the states so the cached file is a check rather
than an input.

**The chance floor** is what a model scores by choosing uniformly among the
candidate pairs on each state. For one state with $\ell$ legal pairs out of $b$
candidates it is $\ell / b$; the reported figure is the mean over states, so
each state weighs equally, because each state is one decision the model was
asked to make.

**The width-blind line** is the harder and more useful reference: what a model
scores if it applies every constraint *except* the grasp-width comparison. It is
computed by raising both arm types' `max_grasp_m` so that grasp can never bind,
re-running the same validator on the same state, and taking $\ell / b'$ where
$b'$ is the number of pairs that survive. So it uses the experiment's own
validator and introduces no new rule or heuristic.

This matters for how the results are read. If a model that loses the declared
width lands near 74.9%, it behaves like a model that lost exactly the width and
kept everything else — which is a much more specific claim than "it got worse".
The line is an **expected score, not a ceiling**: a model can exceed it.

The refusal states have no width-blind line. They contain no legal pair, so
$\ell / b'$ is zero however the width is treated; they are scored by correct
refusal instead, against a floor of its own.

In [20]:
WIDE = 10.0     # metres: an aperture no object in either cast can exceed


def legal_pair_set(probe, width_blind=False):
    """The pairs the deployed validator accepts on this state.

    width_blind=True raises both arm types' grasp aperture first, so the grasp
    comparison can never be the reason a pair is refused, then restores it.  The
    restore is in a finally block because a leaked limit would silently corrupt
    every state computed afterwards in the same process.
    """
    if not width_blind:
        pairs, _ = legal_options(probe)
        return {(t, a) for t, a, _k in pairs}
    saved = {k: CELL.ARM_TYPES[k]["max_grasp_m"] for k in CELL.ARM_TYPES}
    try:
        for k in CELL.ARM_TYPES:
            CELL.ARM_TYPES[k]["max_grasp_m"] = WIDE
        pairs, _ = legal_options(probe)
        return {(t, a) for t, a, _k in pairs}
    finally:
        for k, v in saved.items():
            CELL.ARM_TYPES[k]["max_grasp_m"] = v


def reference_lines(probes):
    """(chance floor, width-blind line, n) over the picking states given."""
    chance, blind = [], []
    for p in probes:
        n_pairs = p["derived"]["n_idle_arms"] * p["derived"]["n_tasks_open"]
        if not n_pairs:
            continue
        legal = legal_pair_set(p)
        # The frozen ground truth and a fresh validator run must agree, or one
        # of the two has drifted and neither can be reported.
        assert len(legal) == p["derived"]["n_legal_pairs"], \
            "recomputed %d legal pairs, probe set stores %d on seq %s" % (
                len(legal), p["derived"]["n_legal_pairs"], p["provenance"]["seq"])
        wb = legal_pair_set(p, width_blind=True)
        assert legal <= wb, "ignoring a constraint can only add options"
        chance.append(len(legal) / n_pairs)
        if wb:
            blind.append(len(legal) / len(wb))
    return (100 * statistics.mean(chance), 100 * statistics.mean(blind), len(chance))


REF = {}
REF["picking"] = reference_lines(POPULATIONS["picking"])
REF["grasp_picking"] = reference_lines(POPULATIONS["grasp_picking"])

# The refusal floor is the chance of correctly declining: one decline weighed
# against every candidate pair the state offers.
refusal_floor = 100 * statistics.mean(
    1.0 / (p["derived"]["n_idle_arms"] * p["derived"]["n_tasks_open"] + 1)
    for p in POPULATIONS["refusal"])

rows = [["Picking states", N["picking"], "%.1f" % REF["picking"][0],
         "%.1f" % REF["picking"][1]],
        [r"\quad Grasp-binding", N["grasp_picking"],
         "%.1f" % REF["grasp_picking"][0], "%.1f" % REF["grasp_picking"][1]],
        ["No-legal-arm states", N["refusal"], "%.1f" % refusal_floor, "{--}"]]
show(["Subset", "States", "Chance", "Width-blind"], rows, "Table 4.6, rebuilt")

# The cached pipeline output must agree with what was just recomputed.
cached = json.load(open(os.path.join(ROOT, "out/ex1_chance_floor.json")))
assert round(100 * cached["uniform"]["mean"], 1) == round(REF["picking"][0], 1)
assert round(100 * cached["width_blind"]["mean"], 1) == round(REF["picking"][1], 1)
g = cached["by_cause"]["grasp"]
assert round(100 * g["uniform"]["mean"], 1) == round(REF["grasp_picking"][0], 1)
assert round(100 * g["width_blind"]["mean"], 1) == round(REF["grasp_picking"][1], 1)
print("\nrecomputed lines agree with the cached out/ex1_chance_floor.json")

Table 4.6, rebuilt
-------------------  ------  ------  -----------
Subset               States  Chance  Width-blind
-------------------  ------  ------  -----------
Picking states       126     35.5    80.9       
\quad Grasp-binding  96      30.5    74.9       
No-legal-arm states  36      22.0    {--}       
-------------------  ------  ------  -----------

recomputed lines agree with the cached out/ex1_chance_floor.json


In [21]:
tex = "\n".join([
    r"\begin{table}[H]", r"\centering",
    r"\caption{Reference values for Experiment~1, cast~A.}",
    r"\label{tab:ex1:floors}",
    r"\begin{tabular}{@{}l S[table-format=3.0] S[table-format=2.1] "
    r"S[table-format=2.1]@{}}", r"\toprule",
    r"Subset & {States} & {Chance} & {Width-blind} \\", r"\midrule"]
    + ["%s \\\\" % " & ".join(str(c) for c in r) for r in rows]
    + [r"\bottomrule", r"\end{tabular}", r"\end{table}", ""])
open(os.path.join(TEX_OUT, "ex1_floors.tex"), "w").write(tex)
BUILT["tab:ex1:floors"] = {"path": os.path.join(TEX_OUT, "ex1_floors.tex"),
                           "headers": None, "rows": rows}
check_against_thesis("tab:ex1:floors", tex)

WIDTH_BLIND_A, CHANCE_A = REF["grasp_picking"][1], REF["grasp_picking"][0]
print("\nprimary reference lines: width-blind %.1f, chance %.1f"
      % (WIDTH_BLIND_A, CHANCE_A))

tab:ex1:floors         MATCHES the thesis on all 8 numeric cells

primary reference lines: width-blind 74.9, chance 30.5


---
# 11. Table 4.7 — the master results table

**Reports:** grasp-binding legality for every model in every condition, with the
negative control beside it. Every other results table in the chapter is a view
of a subset of these cells.

**Source:** all 24 cast A run files.

**The two columns are measured on different populations, and that is the point.**

| Column | Population | Unit |
|---|---|---|
| Legality | the 96 grasp-binding picking scenes | per-scene majority of three repeats, Wilson 95% interval |
| Neg. control | the 30 grasp-free picking scenes | trial level |

The negative control answers the objection that removing the declared width
might simply have degraded the model's answers in general. Grasp blocks nothing
on those 30 states, so the width cannot change which answers are legal there. A
model whose legality collapses on the 96 while holding at 100% on the 30 has
lost the grasp comparison specifically, not competence generally.

**A note on the units.** The Legality column is scene-majority, as the chapter
reports throughout. The negative control column is trial level. Both are
computed below, so the difference is visible rather than buried, and section 20
flags this as the one place where a single table prints two units.

In [22]:
rows, design_data = [], []
for model in MODELS:
    for cond in ORDER:
        rowset = ROWS[("casta", model, cond)]
        ks, ns = legality_scenes(rowset)                 # the reported figure
        kt, nt = legality_trials(rowset)                 # trial level, for comparison
        kn, nn = legality_trials(rowset, grasp=False)    # negative control, trial level
        kns, nns = legality_scenes(rowset, grasp=False)  # negative control, scene level
        width = "absent" if cond.startswith("nowidth") else "present"
        ident = identity_of(RUNGS[CONDITIONS[cond][2]])
        if cond in CONTROLS:
            width = ident = "--"
        rows.append([MODEL_LABEL[model], CONDITIONS[cond][0], width, ident,
                     fmt_rate(ks, ns), "%.1f" % pct(kn, nn)])
        design_data.append(dict(
            model=MODEL_LABEL[model], cond=cond, condition=CONDITIONS[cond][0],
            width=width, identity=ident,
            scene_k=ks, scene_n=ns, scene_pct=pct(ks, ns),
            trial_k=kt, trial_n=nt, trial_pct=pct(kt, nt),
            neg_trial_pct=pct(kn, nn), neg_scene_pct=pct(kns, nns)))

show(["Model", "Condition", "Width", "Identity", "Legality", "Neg. control"],
     blank_repeats(rows), "Table 4.7, rebuilt")

# Every scene denominator must be 96, or 95 where one scene had no
# majority-eligible repeat.  Anything else means a state went missing.
bad = [(d["model"], d["condition"], d["scene_n"]) for d in design_data
       if d["scene_n"] not in (94, 95, 96)]
assert not bad, "unexpected scene denominators: %s" % bad
print("\nscene denominators in use:",
      sorted(collections.Counter(d["scene_n"] for d in design_data).items()))

Table 4.7, rebuilt
------  ------------------  -------  --------  -------------------  ------------
Model   Condition           Width    Identity  Legality             Neg. control
------  ------------------  -------  --------  -------------------  ------------
Gemini  Full Information    present  true      100.0 [96.2, 100.0]  100.0       
        Anonymous           present  withheld  100.0 [96.2, 100.0]  100.0       
        Swapped Names       present  swapped   100.0 [96.2, 100.0]  100.0       
        No Width            absent   true      71.9 [62.2, 79.9]    100.0       
        No Width + Anon.    absent   withheld  77.1 [67.7, 84.4]    100.0       
        No Width + Swapped  absent   swapped   75.0 [65.5, 82.6]    100.0       
        No Rules            --       --        100.0 [96.2, 100.0]  100.0       
        Legal-Arm Control   --       --        100.0 [96.2, 100.0]  100.0       
GPT     Full Information    present  true      97.9 [92.7, 99.4]    100.0       
        A

In [23]:
caption = ("Experiment 1, cast A. Grasp-binding legality across the crossed "
           "design and the two off-design controls. The reported figure is the "
           "per-scene majority with a Wilson 95\\%% interval, the state being "
           "the unit of analysis. Width-blind reference line %.1f\\%%, chance "
           "floor %.1f\\%%. Neg.\\ control is legality on states where grasp "
           "binds nothing, which localises the effect to the grasp constraint."
           % (WIDTH_BLIND_A, CHANCE_A))
# The scene denominator varies by cell, so the note reports the range the data
# actually shows, and the reason for it.  A scene leaves the denominator when
# every one of its repeats was a decline, since a decline is not a proposal and
# a scene with no proposals has no majority to take.  Established below rather
# than asserted.
lo_n = min(d["scene_n"] for d in design_data)
hi_n = max(d["scene_n"] for d in design_data)

dropped = collections.Counter()
for model in MODELS:
    for cond in ORDER:
        seen = collections.defaultdict(list)
        for r in ROWS[("casta", model, cond)]:
            if is_picking(r) and binds_grasp(r):
                seen[scene_key(r)].append(r.get("result"))
        for outcomes in seen.values():
            if not any(o in PROPOSALS for o in outcomes):
                dropped[(MODEL_LABEL[model], set(outcomes) == {"noop"})] += 1

print("scenes dropped from the denominator, by model and cause")
for (model, all_declines), n in sorted(dropped.items()):
    print("  %-7s %-28s %d" % (model, "every repeat declined" if all_declines
                               else "unscorable reply", n))
assert all(all_declines for (_m, all_declines) in dropped), \
    "a scene dropped for a reason other than declining on every repeat"

note = ("All conditions are three repeats of %d frozen states except "
        "Legal-Arm Control, which is one repeat. Each cell is scored on %d %s "
        "%d grasp-binding scenes: a scene leaves the denominator when every "
        "repeat was a decline, which happens only to GPT."
        % (N["all"], lo_n, "or" if hi_n - lo_n == 1 else "to", hi_n))
tex = latex_table("tab:ex1:design",
                  ["Model", "Condition", "Width", "Identity", "Legality",
                   "Neg. control"],
                  blank_repeats(rows), caption, colspec="llllrr", note=note)
check_against_thesis("tab:ex1:design", tex)

scenes dropped from the denominator, by model and cause
  GPT     every repeat declined        9


tab:ex1:design         MATCHES the thesis on all 96 numeric cells


In [24]:
# Diagnostic, not a published table: the same cells at trial level, and the
# negative control at scene level.  A conclusion that changed between the two
# units would be a finding; none does.
diag = [[d["model"], d["condition"],
         "%.1f" % d["scene_pct"], "%d" % d["scene_n"],
         "%.1f" % d["trial_pct"], "%d" % d["trial_n"],
         "%+.1f" % (d["trial_pct"] - d["scene_pct"]),
         "%.1f" % d["neg_trial_pct"], "%.1f" % d["neg_scene_pct"]]
        for d in design_data]
show(["Model", "Condition", "scene %", "scenes", "trial %", "trials",
      "trial - scene", "neg trial %", "neg scene %"], diag,
     "Diagnostic: the reported unit against the alternative")

worst = max(abs(d["trial_pct"] - d["scene_pct"]) for d in design_data)
print("\nlargest disagreement between the two units: %.1f points" % worst)
print("Every model that is above the width-blind line at trial level is above "
      "it at\nscene level too, and every model below it is below it at both, "
      "so no reading in\nthe chapter depends on which unit is used.")

Diagnostic: the reported unit against the alternative
------  ------------------  -------  ------  -------  ------  -------------  -----------  -----------
Model   Condition           scene %  scenes  trial %  trials  trial - scene  neg trial %  neg scene %
------  ------------------  -------  ------  -------  ------  -------------  -----------  -----------
Gemini  Full Information    100.0    96      100.0    288     +0.0           100.0        100.0      
Gemini  Anonymous           100.0    96      100.0    288     +0.0           100.0        100.0      
Gemini  Swapped Names       100.0    96      100.0    288     +0.0           100.0        100.0      
Gemini  No Width            71.9     96      72.2     288     +0.3           100.0        100.0      
Gemini  No Width + Anon.    77.1     96      75.3     288     -1.7           100.0        100.0      
Gemini  No Width + Swapped  75.0     96      76.0     287     +1.0           100.0        100.0      
Gemini  No Rules            

---
# 12. Table 4.8 — baseline competence

**Reports:** what each model does in the Full Information condition, where every
field needed for a correct decision is supplied. This is the ceiling: a failure
here cannot be blamed on missing information, so it measures whether the model
applies stated rules to stated values.

**Source:** the three Full Information run files.

**Three columns, three populations**, which is why the header carries three
different denominators:

| Column | Population | What it asks |
|---|---|---|
| Grasp-binding states | the 96 | can it allocate where the grasp width matters? |
| Correct refusal | the 36 refusal states | does it decline when no legal arm exists? |
| All picking states | the 126 | can it allocate at all? |

All three are per-scene majorities with Wilson intervals.

In [25]:
rows, baseline_data = [], []
for model in MODELS:
    rowset = ROWS[("casta", model, "full")]
    kg, ng = legality_scenes(rowset)                 # 96 grasp-binding picking
    kr, nr = refusal_scenes(rowset)                  # 36 refusal
    ka, na = legality_scenes_all_picking(rowset)     # 126 picking
    rows.append([MODEL_LABEL[model], fmt_rate(kg, ng), fmt_rate(kr, nr),
                 fmt_rate(ka, na)])
    baseline_data.append((MODEL_LABEL[model], (kg, ng), (kr, nr), (ka, na)))

show(["Model", "Grasp-binding states", "Correct refusal", "All picking states"],
     rows, "Table 4.8, rebuilt")

print("\ndenominators actually used")
for label, g, r, a in baseline_data:
    print("  %-7s grasp-binding n=%d   refusal n=%d   all picking n=%d"
          % (label, g[1], r[1], a[1]))

assert all(r[1] == N["refusal"] for _, _, r, _ in baseline_data), \
    "every model faces all 36 refusal states"
assert all(g[1] in (95, 96) for _, g, _, _ in baseline_data)

Table 4.8, rebuilt
------  --------------------  -----------------  -------------------
Model   Grasp-binding states  Correct refusal    All picking states 
------  --------------------  -----------------  -------------------
Gemini  100.0 [96.2, 100.0]   91.7 [78.2, 97.1]  100.0 [97.0, 100.0]
GPT     97.9 [92.7, 99.4]     88.9 [74.7, 95.6]  98.4 [94.4, 99.6]  
Qwen    77.1 [67.7, 84.4]     0.0 [0.0, 9.6]     76.2 [68.0, 82.8]  
------  --------------------  -----------------  -------------------

denominators actually used
  Gemini  grasp-binding n=96   refusal n=36   all picking n=126
  GPT     grasp-binding n=96   refusal n=36   all picking n=125
  Qwen    grasp-binding n=96   refusal n=36   all picking n=126


The all-picking denominators are **not** identical across models: GPT is scored
on 125 scenes, not 126, because one scene has no majority-eligible repeat. The
thesis header prints a single `n=126` over that column for all three rows. The
cell values are correct as printed; the column header is one state optimistic
for GPT. This is recorded in the audit in section 20 as the one wording fix the
notebook found, and the generated LaTeX below prints the true per-model
denominators in the note rather than silently reproducing the header.

In [26]:
caption = "Baseline competence at Full Information, cast A."
note = ("Percentages are per-state majorities across three repeats; brackets "
        "give Wilson 95\\% intervals. Denominators are "
        + "; ".join("%s $n=%d$, %d, %d" % (lbl, g[1], r[1], a[1])
                    for lbl, g, r, a in baseline_data)
        + " for the three columns respectively.")

# The thesis prints the denominators as a three-line header rather than a note,
# so the LaTeX is emitted in that shape and the note carries the true figures.
tex = "\n".join([
    r"\begin{table}[H]", r"  \centering",
    r"  \caption{%s}" % caption, r"  \label{tab:ex1:baseline}", r"  \small",
    r"  \begin{tabular}{@{}lccc@{}}", r"    \toprule",
    r"    Model & Grasp-binding states & Correct refusal & All picking states \\",
    r"          & $n=%d$               & $n=%d$          & $n=%d$ \\"
    % (N["grasp_picking"], N["refusal"], N["picking"]),
    r"          & Legality (\%)        & Refusals correct (\%) & Legality (\%) \\",
    r"    \midrule"]
    + ["    %s \\\\" % " & ".join(str(c) for c in r) for r in rows]
    + [r"    \bottomrule", r"  \end{tabular}",
       r"  \begin{minipage}{0.94\linewidth}\vspace{2pt}\footnotesize",
       r"    " + note, r"  \end{minipage}", r"\end{table}", ""])
open(os.path.join(TEX_OUT, "ex1_baseline.tex"), "w").write(tex)
BUILT["tab:ex1:baseline"] = {"path": os.path.join(TEX_OUT, "ex1_baseline.tex"),
                             "headers": None, "rows": rows}
check_against_thesis("tab:ex1:baseline", tex)

tab:ex1:baseline       MATCHES the thesis on all 27 numeric cells


---
# 13. Table 4.9 — what the violations were caused by

**Reports:** for three conditions, how many proposals each constraint rejected,
with the legality figure beside them.

**Source:** the `violation_cause` field on every rejected proposal, in nine run
files (three models x Full Information, No Width, No Rules).

**This table joins two different units, deliberately.** The Legality column is
the per-scene majority from Table 4.7. The violation counts are raw trial-level
counts over *all* states, not just the 96 — a count of how many times each
constraint was broken, not a rate. They are not divided by anything, so they
cannot be read as a percentage, and the caption says so.

**What the table is for.** Legality alone shows that removing the width hurts.
Composition shows *what* it hurts. If removing the declared grasp width acts
through the grasp comparison and nothing else, then Gemini's violations should
move from nothing to grasp and stay zero everywhere else — which is the
assertion at the end of this section, and it is the load-bearing evidence for
the width subsection's claim.

In [27]:
COMP_CONDITIONS = ["full", "nowidth", "norules"]

raw = []
for model in MODELS:
    for cond in COMP_CONDITIONS:
        rowset = ROWS[("casta", model, cond)]
        v = violations_by_cause(rowset)
        ks, ns = legality_scenes(rowset)
        raw.append({"model": MODEL_LABEL[model], "cond": cond,
                    "condition": CONDITIONS[cond][0],
                    "legality": pct(ks, ns),
                    "counts": {c: v.get(c, 0) for c in CAUSE_ORDER},
                    "arm_state": v.get("arm_state", 0),
                    "total": sum(v.values())})

# Which cause columns to print: those that fired at least once anywhere.  A
# column of zeros carries no information and payload never fires on this cast.
present = [c for c in ["grasp", "reach", "delicate", "arm_state", "no_route"]
           if any(r["counts"].get(c, r["arm_state"] if c == "arm_state" else 0)
                  for r in raw)]

def count_of(r, c):
    return r["arm_state"] if c == "arm_state" else r["counts"].get(c, 0)

headers = ["Model", "Condition", "Legality"] + present + ["Total"]
rows = [[r["model"], r["condition"], "%.1f" % r["legality"]]
        + [count_of(r, c) for c in present] + [r["total"]] for r in raw]
show(headers, blank_repeats(rows), "Table 4.9, rebuilt")

# The legality column must be the same figure Table 4.7 prints, not a
# recomputation that happens to be close.
for r in raw:
    d = next(x for x in design_data
             if x["model"] == r["model"] and x["cond"] == r["cond"])
    assert round(r["legality"], 1) == round(d["scene_pct"], 1), \
        "%s %s: composition says %.1f, design table says %.1f" % (
            r["model"], r["condition"], r["legality"], d["scene_pct"])

# The load-bearing check: for Gemini, removing the width must move violations
# onto grasp and onto nothing else.
gem = next(r for r in raw if r["model"] == "Gemini" and r["cond"] == "nowidth")
fired = {c for c in present if count_of(gem, c)}
assert fired == {"grasp"}, \
    "Gemini at No Width should violate grasp and nothing else, got %s" % sorted(fired)
print("\nlegality column agrees with Table 4.7 in all nine cells")
print("Gemini at No Width violates: %s" % sorted(fired))

Table 4.9, rebuilt
------  ----------------  --------  -----  -----  --------  ---------  --------  -----
Model   Condition         Legality  grasp  reach  delicate  arm_state  no_route  Total
------  ----------------  --------  -----  -----  --------  ---------  --------  -----
Gemini  Full Information  100.0     0      0      0         0          7         7    
        No Width          71.9      133    0      0         0          0         133  
        No Rules          100.0     1      0      0         0          0         1    
GPT     Full Information  97.9      7      0      2         0          10        19   
        No Width          71.6      121    0      0         0          3         124  
        No Rules          94.7      28     0      3         0          5         36   
Qwen    Full Information  77.1      41     52     39        67         1         200  
        No Width          78.1      43     56     34        55         1         189  
        No Rules        

In [28]:
caption = ("Violations by cause, cast A, with grasp-binding legality "
           "(per-scene majority, \\%) alongside. Removing the declared width "
           "changes the composition on grasp alone; removing the rules "
           "multiplies violations across every constraint. Violation counts "
           "are over all trials.")
tex = latex_table("tab:ex1:composition", headers, blank_repeats(rows), caption,
                  colspec="ll" + "r" * (len(headers) - 2))
check_against_thesis("tab:ex1:composition", tex)

tab:ex1:composition    MATCHES the thesis on all 63 numeric cells


---
# 14. Table 4.10 — three signatures of an unregistered loss

**Reports:** whether a model shows any sign of having noticed that the declared
width is gone, read three ways, Full Information against No Width.

**Source:** the Full Information and No Width run files, plus the No Width +
Anonymous files for the reasons count.

The argument the table serves: a model that registered the loss would **say so**,
**abstain more**, or both. If it does neither, and instead spreads its answers
across the arms as though every object fits, then the width was not being
reasoned about at all — it was being read.

| Signature | Measure | Unit |
|---|---|---|
| Reasons admitting the gap | free-text reasons on grasp violations that mention missing width | trial level, raw count |
| Correct refusal | declines over the 36 refusal states | per-scene majority |
| Franka share | proposals sent to a Franka arm | per proposal, trial level |

Three units in one table. The chapter's rule is that where a table cannot be
forced to one unit, each unit is named in the caption rather than the numbers
distorted to match — and the caption does name them.

**The reasons regex is deliberately generous.** It matches "unknown",
"unstated", "missing", "not provided", "absent" and several more. A loose
pattern that still finds nothing is stronger evidence than a strict one that
finds nothing, because the loose pattern would have caught a partial admission.

**Why Franka share is the substitution measure.** The cell has two Franka arms
and two UR arms. The Franka has the narrower aperture, so under Full Information
a model that respects the width sends *less* than half its proposals there — and
each does, at 28.1%, 26.0% and 21.0%. If the width is gone and a model has no
substitute for it, its share drifts towards 50%, which is what treating the four
arms as interchangeable looks like. Gemini and GPT both move to an interval that
contains 50.0; Qwen does not move, having never used the width in the first
place.

In [29]:
# Deliberately generous: any of these words in a reason counts as admitting the
# width is missing.
MISSING_INFO = re.compile(
    r"(unknown|unstated|not stated|missing|unspecified|no (?:declared )?width"
    r"|not (?:given|provided|specified|declared)|absent)", re.I)

rows, sig_data = [], []
for model in MODELS:
    shown = ROWS[("casta", model, "full")]
    absent = ROWS[("casta", model, "nowidth")]

    # 1. Does it say so?  Counted over BOTH width-absent design conditions, so
    #    the denominator is larger than any single cell.
    admit = examined = 0
    for cond in ("nowidth", "nowidth-anon"):
        for r in ROWS[("casta", model, cond)]:
            if r.get("violation_cause") != "grasp":
                continue
            examined += 1
            admit += bool(MISSING_INFO.search(r.get("model_reason") or ""))

    # 2. Does it act so?
    rb, nb = refusal_scenes(shown)
    rm, nm = refusal_scenes(absent)

    # 3. What does it do instead?
    fb, nfb = franka_share(shown)
    fm, nfm = franka_share(absent)

    rows += [[MODEL_LABEL[model], "Reasons admitting the gap", "--",
              "%d/%d" % (admit, examined), "--"],
             ["", "Correct refusal", "%.1f" % pct(rb, nb), "%.1f" % pct(rm, nm),
              fmt_diff((rb, nb), (rm, nm))],
             ["", "Franka share", "%.1f" % pct(fb, nfb), fmt_rate(fm, nfm),
              fmt_diff((fm, nfm), (fb, nfb))]]
    sig_data.append(dict(model=MODEL_LABEL[model], admit=admit, examined=examined,
                         franka_absent=(fm, nfm)))

show(["Model", "Signature", "Width shown", "Width absent", "Change [95% CI]"],
     rows, "Table 4.10, rebuilt")

pooled_admit = sum(d["admit"] for d in sig_data)
pooled_examined = sum(d["examined"] for d in sig_data)
assert pooled_admit == 0, "a reason admitting the gap would change the argument"
print("\nreasons examined %d, admitting the missing width %d"
      % (pooled_examined, pooled_admit))
print("one-sided 95%% upper bound on the pooled rate: %.2f%% (rule of three)"
      % (300.0 / pooled_examined))
for d in sig_data:
    lo, hi = wilson(*d["franka_absent"])
    print("  %-7s Franka share width absent %.1f [%.1f, %.1f]  %s %.1f"
          % (d["model"], pct(*d["franka_absent"]), lo, hi,
             "includes" if lo <= ARM_INDIFFERENT <= hi else "excludes",
             ARM_INDIFFERENT))

Table 4.10, rebuilt
------  -------------------------  -----------  -----------------  --------------------
Model   Signature                  Width shown  Width absent       Change [95% CI]     
------  -------------------------  -----------  -----------------  --------------------
Gemini  Reasons admitting the gap  --           0/256              --                  
        Correct refusal            91.7         50.0               +41.7 [+21.1, +58.1]
        Franka share               28.1         48.0 [43.3, 52.7]  +20.0 [+13.4, +26.3]
GPT     Reasons admitting the gap  --           0/237              --                  
        Correct refusal            88.9         52.8               +36.1 [+15.3, +53.2]
        Franka share               26.0         45.7 [41.0, 50.6]  +19.7 [+13.1, +26.1]
Qwen    Reasons admitting the gap  --           0/85               --                  
        Correct refusal            0.0          0.0                +0.0 [-9.6, +9.6]   
        Fran

In [30]:
caption = ("Three readings of the same question, cast A, Full Information "
           "against No Width. A model that registered the loss of the declared "
           "width would say so, abstain more, or both. Refusal and Franka share "
           "are per-scene and per-proposal respectively. The refusal change is "
           "Full Information minus No Width, so a positive value means the "
           "model abstains less when it knows less. Two Franka arms and two UR "
           "arms, so a Franka share of %.1f\\%% is what treating the four arms "
           "as interchangeable looks like." % ARM_INDIFFERENT)
note = ("Reasons are counted over both width-absent design conditions, so the "
        "denominator is larger than a single cell. The one-sided 95\\% upper "
        "bound on the pooled rate is given in the text.")
tex = latex_table("tab:ex1:signatures",
                  ["Model", "Signature", "Width shown", "Width absent",
                   "Change [95\\% CI]"],
                  rows, caption, colspec="llrrr", note=note)
check_against_thesis("tab:ex1:signatures", tex)

tab:ex1:signatures     MATCHES the thesis on all 42 numeric cells


---
# 15. Table 4.11 — cast B, the second object set

**Reports:** whether the width effect appears on a different set of objects.

**Source:** the three cast B run files, the three independent single-repeat cast
B runs, and `probes/ex1_setb_v1.json` for the reference lines.

**This is a generalisation check, not a replication**, and the table is built to
make that impossible to overstate. Cast B is ten objects disjoint from cast A,
108 states, GPT only, run at one model for cost reasons. It answers one question:
does the direction of the effect survive a change of objects? It does. The
magnitude does not — the cast A gap is roughly three times the cast B gap — and
the two intervals do not overlap.

**The independent-run column** is a second, separate single-repeat execution of
the same three cells at the same prompt version on the same states. Its trials
exist nowhere else: GPT answered 25, 19 and 21 of the 108 states differently from
repeat 1 of the three-repeat run. It is reported as a run-to-run stability check
and **not pooled** with the three-repeat run, because cast A is three repeats
throughout and a four-repeat cast B would need a tie rule to buy 0.2 of a point.

Note that this table reports the trial-level rate in its Legality column, with
the scene majority in its own column beside it, because the run-to-run column it
is being compared against is a single repeat and has no scene majority of its own.

In [31]:
PROBES_B = PROBE_SET["castb"]["probes"]
REF["castb_grasp"] = reference_lines(
    [p for p in PROBES_B if is_picking(p) and binds_grasp(p)])
WIDTH_BLIND_B, CHANCE_B = REF["castb_grasp"][1], REF["castb_grasp"][0]
print("cast B reference lines: width-blind %.1f, chance %.1f  (n=%d states)"
      % (WIDTH_BLIND_B, CHANCE_B, REF["castb_grasp"][2]))

cached_b = json.load(open(os.path.join(ROOT, "out/ex1_setb_floors.json")))
assert round(100 * cached_b["by_cause"]["grasp"]["width_blind"]["mean"], 1) \
       == round(WIDTH_BLIND_B, 1)
assert round(100 * cached_b["by_cause"]["grasp"]["uniform"]["mean"], 1) \
       == round(CHANCE_B, 1)
print("recomputed cast B lines agree with out/ex1_setb_floors.json")

cast B reference lines: width-blind 69.2, chance 34.6  (n=58 states)
recomputed cast B lines agree with out/ex1_setb_floors.json


In [32]:
# THE UNIT IS THE SCENE. The chapter reports the per-scene majority
# throughout, and Table 4.11's contrasts are compared directly with the cast A
# contrasts, so both must be at the same denominator. This cell emitted the
# TRIAL-level rate with a separate n column until 2026-09-10 -- the same defect
# section 11 records for Table 4.7 -- which is four numbers the thesis does not
# print and a Legality column it does not use.
rows, base = [], None
for cond in CASTB_CONDITIONS:
    scenes = legality_scenes(ROWS[("castb", "gpt", cond)])
    gap = "--" if base is None else fmt_diff(base, scenes)
    if base is None:
        base = scenes
    # One repeat, so the independent run's scene majority is its trial rate.
    solo = legality_scenes(ROWS[("castb_r1", "gpt", cond)])
    rows.append([CONDITIONS[cond][0], fmt_rate(*scenes), scenes[1],
                 "%.1f" % pct(*solo), gap])

show(["Condition", "Legality", "Scenes", "Independent run",
      "Gap from Full Info"], rows, "Table 4.11, rebuilt")

# The comparison the subsection rests on, at the unit the chapter reports.
a = newcombe(*legality_scenes(ROWS[("casta", "gpt", "full")]),
             *legality_scenes(ROWS[("casta", "gpt", "nowidth")]))
b = newcombe(*legality_scenes(ROWS[("castb", "gpt", "full")]),
             *legality_scenes(ROWS[("castb", "gpt", "nowidth")]))
print("\nwidth-removal gap, GPT, per-scene majority")
print("  cast A  %+.1f [%+.1f, %+.1f]   width-blind line %.1f" % (a + (WIDTH_BLIND_A,)))
print("  cast B  %+.1f [%+.1f, %+.1f]   width-blind line %.1f" % (b + (WIDTH_BLIND_B,)))
assert not spans_zero(a[1], a[2]) and not spans_zero(b[1], b[2]), \
    "the direction is the claim; both gaps must exclude zero"
print("  both exclude zero, so the direction reproduces")
print("  intervals overlap: %s" % (not (a[1] > b[2] or b[1] > a[2])))

# The independent run is a separate execution, not a slice of the three-repeat
# run.  Check that by comparing its answers against repeat 1 of the main run.
for cond in CASTB_CONDITIONS:
    r1 = {scene_key(r): r.get("result") for r in ROWS[("castb_r1", "gpt", cond)]}
    main1 = {scene_key(r): r.get("result")
             for r in ROWS[("castb", "gpt", cond)] if r.get("repeat") == 1}
    differ = sum(1 for k in r1 if r1[k] != main1.get(k))
    print("  %-19s independent run differs from repeat 1 on %d of %d states"
          % (CONDITIONS[cond][0], differ, len(r1)))
    assert differ > 0, "an identical run would mean the file is a copy, not a rerun"

Table 4.11, rebuilt
----------------  -------------------  ------  ---------------  -------------------
Condition         Legality             Scenes  Independent run  Gap from Full Info 
----------------  -------------------  ------  ---------------  -------------------
Full Information  100.0 [93.8, 100.0]  58      98.1             --                 
Anonymous         100.0 [93.8, 100.0]  58      100.0            +0.0 [-6.2, +6.2]  
No Width          87.5 [76.4, 93.8]    56      88.5             +12.5 [+3.6, +23.6]
No Width + Anon.  86.0 [74.7, 92.7]    57      92.2             +14.0 [+4.9, +25.3]
----------------  -------------------  ------  ---------------  -------------------

width-removal gap, GPT, per-scene majority
  cast A  +26.3 [+16.7, +36.2]   width-blind line 74.9
  cast B  +12.5 [+3.6, +23.6]   width-blind line 69.2
  both exclude zero, so the direction reproduces
  intervals overlap: True
  Full Information    independent run differs from repeat 1 on 14 of 108 states


In [33]:
caption = ("Cast B, ten objects disjoint from cast A, GPT only, three repeats "
           "on %d states. Width-blind reference line %.1f\\%%, chance floor "
           "%.1f\\%%. The direction of the width effect reproduces and the "
           "magnitude does not." % (len(PROBES_B), WIDTH_BLIND_B, CHANCE_B))
note = ("Reported as a generalisation check and not a replication. Cast B was "
        "run at one model for cost reasons, which is stated as a limitation "
        "rather than presented as a second experiment. The independent-run "
        "column is a separate single-repeat execution of the same four cells "
        "at the same prompt version. Its trials are not contained in the "
        "three-repeat run and it is not pooled with it, because cast A is "
        "three repeats throughout.")
tex = latex_table("tab:ex1:castb",
                  ["Condition", "Legality", "Scenes", "Independent run",
                   "Gap from Full Info"],
                  rows, caption, colspec="lrrrr", note=note)
check_against_thesis("tab:ex1:castb", tex)

tab:ex1:castb          MATCHES the thesis on all 29 numeric cells


---
# 16. Table F.1 — the state counts quoted in the glossary

**Reports:** the vocabulary the chapter uses, with the state counts that appear
inside three of the definitions.

**Source:** `probes/ex1_v2.json`. The prose of the glossary is written by hand;
what is checked here is that the counts embedded in it are the ones the probe set
actually holds, since those definitions are what every denominator in the chapter
refers back to.

In [34]:
import re

counts = {
    "states in the probe set": N["all"],
    "picking states": N["picking"],
    "refusal states": N["refusal"],
    "grasp-binding states": N["grasp_binding"],
    "grasp-binding picking states": N["grasp_picking"],
    "grasp-binding refusal states": N["grasp_refusal"],
}
show(["Quantity", "States"], [[k, v] for k, v in counts.items()],
     "Counts quoted inside the Appendix F definitions")

published = thesis_source("tab:ex1:terms")
if published is None:
    CHECKS.append(("tab:ex1:terms", "SKIPPED", "thesis source not found"))
    print("\ntab:ex1:terms          SKIPPED")
else:
    quoted = set(table_numbers(published))
    # Only the counts the glossary actually quotes need to appear; the table is
    # prose and carries no other numbers.
    needed = {float(N["refusal"]), float(N["grasp_picking"])}
    missing = needed - quoted
    assert not missing, ("Appendix F quotes state counts that the probe set "
                         "does not support: %s" % sorted(missing))
    # And the rule that makes the denominators legible must still be stated.
    # Matched on MEANING, not on one phrasing: this asserted the literal string
    # "not in the denominator" until 2026-09-10, and broke when the appendix was
    # reworded to "declines excluded from the denominator" -- a check that fails
    # on a synonym tests the sentence, not the claim.
    decline_rule = re.search(
        r"declines?\b[^.]*\b(excluded from|not in|outside)\b[^.]*denominator"
        r"|denominator[^.]*\bexclud\w*[^.]*\bdeclines?\b",
        published, re.I | re.S)
    assert decline_rule, ("Appendix F must keep the sentence stating that "
                          "declines are excluded from the denominator")
    CHECKS.append(("tab:ex1:terms", "MATCHES",
                   "quoted state counts and the decline rule"))
    print("\ntab:ex1:terms          MATCHES the thesis on its quoted state counts")

Counts quoted inside the Appendix F definitions
----------------------------  ------
Quantity                      States
----------------------------  ------
states in the probe set       162   
picking states                126   
refusal states                36    
grasp-binding states          122   
grasp-binding picking states  96    
grasp-binding refusal states  26    
----------------------------  ------

tab:ex1:terms          MATCHES the thesis on its quoted state counts


---
# 17. Table G.1 — the object-name swap

**Reports:** the pre-registered directional test on whether an object's *name*
can supply what its declared width supplied.

**Source:** the four run files per model (Full Information, Swapped Names, No
Width, No Width + Swapped) and `probes/ex1_v2.json` for the true object behind
each task. The swap definition is imported from `experiments/ex1/mislabel.py` —
the module that performed the swap — rather than restated here.

**The manipulation.** Three pairs of objects exchange names across the Franka's
80 mm aperture: each pair is category-matched, neither object is delicate, and
one of the two is wider than the aperture while the other is narrower. **The
physical fields stay with the true object**, so the ground truth does not move
and a legal answer stays legal. Only the name is wrong. Four further objects are
left untouched inside the same prompts, as an in-prompt control for drift.

**Why the statistic is a separation and not an interval on either column.**
If a model is following the *name*, the two swapped groups must move in
**opposite** directions: an object that is genuinely wide but now carries a
narrow name should be sent to the Franka more often, and an object that is
genuinely narrow but now carries a wide name should be sent there less often.
Both columns moving the same way is drift, not name-following, however far
either moves. So the test is the gap between the two columns, benchmarked
against the drift on the four objects nobody touched.

**Why it is in an appendix.** Only about 6% of probe tasks offer a legal choice
between a Franka and a UR arm, so this test is badly underpowered and cannot
carry the identity finding on its own. The body of the chapter leads instead on
legality under the false name, where Gemini scores 288 of 288. This table is the
directional check behind that argument, reported in full and reported as
underpowered.

In [35]:
from experiments.ex1.mislabel import SWAP_PAIRS, CONTROLS as SWAP_CONTROLS, \
                                     APERTURE_FRANKA

# SWAP_PAIRS is a list of (object_a, object_b) that exchange names.  What the
# analysis needs is each swapped object with its TRUE width, so the direction it
# was moved across the aperture can be read off.
swapped_objects = []
for a, b in SWAP_PAIRS:
    for name in (a, b):
        swapped_objects.append((name, spec_of(name)["grasp_m"]))

print("Franka aperture %.3f m" % APERTURE_FRANKA)
print("\nswapped pairs (physical fields stay with the true object)")
for a, b in SWAP_PAIRS:
    print("  %-18s %.3f m  <->  %-18s %.3f m"
          % (a.replace("ycb_", ""), spec_of(a)["grasp_m"],
             b.replace("ycb_", ""), spec_of(b)["grasp_m"]))
print("\nuntouched in-prompt controls: %s"
      % ", ".join(c.replace("ycb_", "") for c in SWAP_CONTROLS))

# Each pair must straddle the aperture, or the swap moves no name across the
# boundary that matters and the test is empty.
for a, b in SWAP_PAIRS:
    wa, wb = spec_of(a)["grasp_m"], spec_of(b)["grasp_m"]
    assert (wa > APERTURE_FRANKA) != (wb > APERTURE_FRANKA), \
        "%s and %s sit on the same side of the aperture" % (a, b)
print("\nevery swapped pair straddles the aperture")

Franka aperture 0.080 m

swapped pairs (physical fields stay with the true object)
  large_clamp        0.122 m  <->  power_drill        0.050 m
  mustard            0.096 m  <->  soup_can           0.068 m
  meat_can           0.084 m  <->  gelatin_box        0.073 m

untouched in-prompt controls: mug, mug2, banana, bowl

every swapped pair straddles the aperture


In [36]:
# Which object each task was really about, so a proposal can be attributed to the
# true object rather than to the name the prompt showed.
true_object = {}
for p in PROBES_A:
    pv = p["provenance"]
    for t in p["state"]["tasks"]:
        true_object[(pv["source"], pv["seq"], pv["round"], t["id"])] = t["object"]


def franka_share_by_object(rows):
    """(franka proposals, all proposals) per true object."""
    sent, total = collections.Counter(), collections.Counter()
    for r in rows:
        dec = r.get("decision") or {}
        arm, tid = dec.get("arm"), dec.get("task_id")
        if not arm or tid is None:
            continue
        pv = r["provenance"]
        obj = true_object.get((pv["source"], pv["seq"], pv["round"], tid))
        if obj is None:
            continue
        total[obj] += 1
        sent[obj] += arm.startswith("franka")
    return sent, total


rows = []
for model in MODELS:
    # Each swapped condition is compared against the condition it was built
    # from: No Width + Swapped descends from No Width, not from No Width +
    # Anonymous, because the swapped condition carries names that are present
    # and false rather than withheld.
    for base, manip, label in (("full", "swap", "width present"),
                               ("nowidth", "nowidth-swap", "width absent")):
        fb, tb = franka_share_by_object(ROWS[("casta", model, base)])
        fs, ts = franka_share_by_object(ROWS[("casta", model, manip)])

        groups = {"wide": [0, 0, 0, 0], "narrow": [0, 0, 0, 0],
                  "control": [0, 0, 0, 0]}   # [swapped k, swapped n, base k, base n]
        for obj, width in swapped_objects:
            g = groups["wide" if width > APERTURE_FRANKA else "narrow"]
            g[0] += fs[obj]; g[1] += ts[obj]; g[2] += fb[obj]; g[3] += tb[obj]
        for obj in SWAP_CONTROLS:
            g = groups["control"]
            g[0] += fs[obj]; g[1] += ts[obj]; g[2] += fb[obj]; g[3] += tb[obj]

        change = {k: newcombe(*v) for k, v in groups.items()}
        separation = diff_of_diffs(change["wide"], change["narrow"])

        # The reported finding is that no separation is detectable.  Assert it,
        # so a change in the data that overturned it could not pass unnoticed.
        assert spans_zero(separation[1], separation[2]), \
            "%s %s: separation no longer spans zero" % (model, label)

        rows.append([MODEL_LABEL[model], label,
                     "%+.1f" % change["wide"][0], "%+.1f" % change["narrow"][0],
                     "%+.1f" % change["control"][0],
                     "%+.1f [%+.1f, %+.1f]" % separation])

show(["Model", "Width", "Wide obj, narrow name", "Narrow obj, wide name",
      "Untouched controls", "Separation"], blank_repeats(rows),
     "Table G.1, rebuilt")
print("\nEvery separation spans zero: no model follows the name.")

Table G.1, rebuilt
------  -------------  ---------------------  ---------------------  ------------------  -------------------
Model   Width          Wide obj, narrow name  Narrow obj, wide name  Untouched controls  Separation         
------  -------------  ---------------------  ---------------------  ------------------  -------------------
Gemini  width present  +0.0                   -5.3                   +0.6                +5.3 [-8.3, +19.0] 
        width absent   -5.9                   -6.8                   +0.6                +0.8 [-15.9, +17.5]
GPT     width present  +1.1                   -6.7                   -2.2                +7.8 [-7.4, +23.0] 
        width absent   -3.4                   -1.6                   +0.3                -1.8 [-17.7, +14.0]
Qwen    width present  +2.3                   -3.0                   +4.7                +5.3 [-4.7, +15.4] 
        width absent   +1.0                   -6.2                   +7.1                +7.2 [-3.3, +17.8] 


In [37]:
caption = ("The pre-registered directional test on the object-name swap, cast "
           "A. Change in Franka share, swapped condition minus its matched "
           "unmanipulated parent, pooled by the direction the name was moved "
           "across the Franka aperture. Name-following requires the two "
           "swapped columns to move in OPPOSITE directions, so the separation "
           "between them is the statistic rather than whether either column "
           "excludes zero. The untouched controls are the benchmark.")
note = ("Three pairs swap across the aperture, category-matched, none "
        "delicate. Physical fields stay with the true object, so ground truth "
        "does not move. Four objects are left unmanipulated as in-prompt "
        "controls. The parent of No Width + Swapped is No Width, not No Width "
        "+ Anonymous, because the swapped condition carries names that are "
        "present but false.")
tex = "\n".join([
    r"\begin{table}[H]", r"  \centering",
    r"  \caption{%s}" % caption, r"  \label{tab:ex1:swap}", r"  \footnotesize",
    r"  \setlength{\tabcolsep}{5pt}",
    r"  \begin{tabular}{llrrrr}", r"    \toprule",
    r"    Model & Width & {Wide obj,} & {Narrow obj,} & {Untouched} & {Separation} \\",
    r"          &       & {narrow name} & {wide name} & {controls} & \\",
    r"    \midrule"]
    + ["    %s \\\\" % " & ".join(str(c) for c in r) for r in blank_repeats(rows)]
    + [r"    \bottomrule", r"  \end{tabular}",
       r"  \begin{minipage}{0.84\linewidth}\vspace{2pt}\footnotesize " + note
       + r"\end{minipage}", r"\end{table}", ""])
open(os.path.join(TEX_OUT, "ex1_swap.tex"), "w").write(tex)
BUILT["tab:ex1:swap"] = {"path": os.path.join(TEX_OUT, "ex1_swap.tex"),
                         "headers": None, "rows": rows}
check_against_thesis("tab:ex1:swap", tex)

tab:ex1:swap           MATCHES the thesis on all 36 numeric cells


---
# 18. The contrasts the prose quotes

Several numbers appear in sentences rather than in tables. They are computed
here for the same reason the tables are: so that the sentence and the number
cannot drift apart.

All of these are Newcombe intervals on two **scene-majority** rates, which is
the unit the chapter reports. The trial-level versions of the same contrasts are
printed beside them, because the earlier drafts of this chapter quoted trial-era
figures and anyone comparing against an old draft needs to see both.

In [38]:
CONTRASTS = [
    ("Width removal", "names true",       "full",    "nowidth"),
    ("Width removal", "names anonymised", "anon",    "nowidth-anon"),
    ("Width removal", "names swapped",    "swap",    "nowidth-swap"),
    ("Identity removed", "width present", "full",    "anon"),
    ("Identity removed", "width absent",  "nowidth", "nowidth-anon"),
    ("Identity swapped", "width present", "full",    "swap"),
    ("Identity swapped", "width absent",  "nowidth", "nowidth-swap"),
    ("Rules removed", "width present",    "full",    "norules"),
]

rows = []
for family, level, base, manip in CONTRASTS:
    row = [family, level]
    for model in MODELS:
        b = legality_scenes(ROWS[("casta", model, base)])
        m = legality_scenes(ROWS[("casta", model, manip)])
        row.append(fmt_diff(b, m))
    rows.append(row)
show(["Family", "Level"] + [MODEL_LABEL[m] for m in MODELS],
     blank_repeats(rows), "Contrasts at the reported unit (scene majority)")

# The three figures the width subsection quotes verbatim.
quoted = {}
for model in MODELS:
    b = legality_scenes(ROWS[("casta", model, "full")])
    m = legality_scenes(ROWS[("casta", model, "nowidth")])
    quoted[MODEL_LABEL[model]] = newcombe(*b, *m)
print("\nquoted in the width subsection")
for name, (d, lo, hi) in quoted.items():
    print("  %-7s %+.1f [%+.1f, %+.1f]" % (name, d, lo, hi))
assert (round(quoted["Gemini"][0], 1), round(quoted["GPT"][0], 1),
        round(quoted["Qwen"][0], 1)) == (28.1, 26.3, -1.0), \
    "the width-removal gaps quoted in the chapter have moved"

Contrasts at the reported unit (scene majority)
----------------  ----------------  --------------------  --------------------  -------------------
Family            Level             Gemini                GPT                   Qwen               
----------------  ----------------  --------------------  --------------------  -------------------
Width removal     names true        +28.1 [+19.2, +37.8]  +26.3 [+16.7, +36.2]  -1.0 [-12.8, +10.7]
                  names anonymised  +22.9 [+14.7, +32.3]  +21.1 [+12.5, +30.4]  +3.1 [-9.2, +15.3] 
                  names swapped     +25.0 [+16.5, +34.5]  +16.8 [+7.6, +26.4]   -2.1 [-14.2, +10.1]
Identity removed  width present     +0.0 [-3.8, +3.8]     -1.0 [-6.3, +3.9]     +1.0 [-10.9, +13.0]
                  width absent      -5.2 [-17.3, +7.1]    -6.3 [-18.4, +6.0]    +5.2 [-7.0, +17.2] 
Identity swapped  width present     +0.0 [-3.8, +3.8]     +2.1 [-3.7, +8.4]     +3.1 [-9.0, +15.1] 
                  width absent      -3.1 [-15.4, +9.

In [39]:
# Where the width-removed cells land relative to the width-blind line: the
# claim is that Gemini and GPT fall TO it, not merely below Full Information.
print("Width-absent cells against the width-blind line of %.1f\n" % WIDTH_BLIND_A)
print("%-7s %-19s %-21s %s" % ("model", "condition", "legality", "line inside interval"))
for model in MODELS:
    for cond in ("nowidth", "nowidth-anon", "nowidth-swap"):
        k, n = legality_scenes(ROWS[("casta", model, cond)])
        lo, hi = wilson(k, n)
        print("%-7s %-19s %-21s %s"
              % (MODEL_LABEL[model], CONDITIONS[cond][0], fmt_rate(k, n),
                 "yes" if lo <= WIDTH_BLIND_A <= hi else "no"))

# The interaction: does the size of the width effect depend on what identity
# information was available?
print("\nInteraction, width gap by identity level")
for model in MODELS:
    gap = {}
    for level, base, manip in (("true", "full", "nowidth"),
                               ("anon", "anon", "nowidth-anon"),
                               ("swap", "swap", "nowidth-swap")):
        gap[level] = newcombe(*legality_scenes(ROWS[("casta", model, base)]),
                              *legality_scenes(ROWS[("casta", model, manip)]))
    va = diff_of_diffs(gap["true"], gap["anon"])
    vs = diff_of_diffs(gap["true"], gap["swap"])
    print("  %-7s vs anonymisation %+.1f [%+.1f, %+.1f]   vs swap %+.1f [%+.1f, %+.1f]"
          % ((MODEL_LABEL[model],) + va + vs))
    assert spans_zero(va[1], va[2]) and spans_zero(vs[1], vs[2]), \
        "the chapter reports no detectable interaction"
print("Every interval spans zero, so no interaction is detectable at this "
      "resolution.\nThat is not the same as no interaction existing.")

Width-absent cells against the width-blind line of 74.9

model   condition           legality              line inside interval
Gemini  No Width            71.9 [62.2, 79.9]     yes
Gemini  No Width + Anon.    77.1 [67.7, 84.4]     yes
Gemini  No Width + Swapped  75.0 [65.5, 82.6]     yes
GPT     No Width            71.6 [61.8, 79.7]     yes
GPT     No Width + Anon.    77.9 [68.6, 85.1]     yes
GPT     No Width + Swapped  78.9 [69.7, 85.9]     yes
Qwen    No Width            78.1 [68.9, 85.2]     yes
Qwen    No Width + Anon.    72.9 [63.3, 80.8]     yes
Qwen    No Width + Swapped  76.0 [66.6, 83.5]     yes

Interaction, width gap by identity level
  Gemini  vs anonymisation +5.2 [-7.6, +18.0]   vs swap +3.1 [-9.8, +16.1]
  GPT     vs anonymisation +5.3 [-8.0, +18.5]   vs swap +9.5 [-4.1, +23.1]
  Qwen    vs anonymisation -4.2 [-21.1, +12.8]   vs swap +1.0 [-15.9, +18.0]
Every interval spans zero, so no interaction is detectable at this resolution.
That is not the same as no interaction

In [40]:
# The image-on robustness sentence: GPT saw a rendered frame as well as the
# text, in the No Width + Anonymous condition.  Its matched comparison is the
# text-only cell of the SAME condition, since that one withholds names too.
img = ROWS[("image", "gpt", "nowidth-anon")]
txt = ROWS[("casta", "gpt", "nowidth-anon")]
assert {r.get("rung") for r in img} == {CONDITIONS["nowidth-anon"][2]}, \
    "the image run must be the No Width + Anonymous condition"

ki, ni = legality_scenes(img)
kt, nt = legality_scenes(txt)
d = newcombe(ki, ni, kt, nt)
print("Image on, GPT, No Width + Anonymous")
print("  image on   %s  (n=%d scenes)" % (fmt_rate(ki, ni), ni))
print("  text only  %s  (n=%d scenes)" % (fmt_rate(kt, nt), nt))
print("  difference %+.1f [%+.1f, %+.1f]  %s"
      % (d + ("spans zero" if spans_zero(d[1], d[2]) else "EXCLUDES ZERO",)))
assert spans_zero(d[1], d[2]), \
    "the chapter reports the image making no measurable difference"

Image on, GPT, No Width + Anonymous
  image on   74.0 [64.4, 81.7]  (n=96 scenes)
  text only  77.9 [68.6, 85.1]  (n=95 scenes)
  difference -3.9 [-15.9, +8.2]  spans zero


---
# 19. Everything that was generated

The LaTeX written by this notebook, beside the file the thesis currently inputs.
For the six tables the thesis keeps in its own files, these are drop-in
replacements. The other seven are written inline in `main.tex` and the generated
file is the reference to paste against.

In [41]:
print("written to %s\n" % TEX_OUT)
for label, info in BUILT.items():
    stem = label.replace("tab:", "").replace(":", "_")
    in_thesis = os.path.join(THESIS_REPO, "tables", stem + ".tex")
    where = ("thesis tables/%s.tex" % stem if os.path.exists(in_thesis)
             else "inline in main.tex")
    print("  %-22s -> %-24s  (thesis keeps it %s)"
          % (label, os.path.basename(info["path"]), where))

written to /Users/erinsarlak/Downloads/MastersDissertation/fourarm/tables/ex1

  tab:ex1:worked         -> ex1_worked.tex            (thesis keeps it inline in main.tex)
  tab:ex1:binding        -> ex1_binding.tex           (thesis keeps it inline in main.tex)
  tab:ex1:subsets        -> ex1_subsets.tex           (thesis keeps it inline in main.tex)
  tab:ex1:rungs          -> ex1_rungs.tex             (thesis keeps it inline in main.tex)
  tab:ex1:grid           -> ex1_grid.tex              (thesis keeps it inline in main.tex)
  tab:ex1:floors         -> ex1_floors.tex            (thesis keeps it inline in main.tex)
  tab:ex1:design         -> ex1_design.tex            (thesis keeps it thesis tables/ex1_design.tex)
  tab:ex1:baseline       -> ex1_baseline.tex          (thesis keeps it thesis tables/ex1_baseline.tex)
  tab:ex1:composition    -> ex1_composition.tex       (thesis keeps it thesis tables/ex1_composition.tex)
  tab:ex1:signatures     -> ex1_signatures.tex        (thesis kee

---
# 20. Reproducibility audit

One row per Experiment 1 table: where its numbers come from, which section of
this notebook builds it, and whether the generated table matches what the thesis
currently prints.

In [42]:
AUDIT = [
    # (thesis number, label, section, source data, what it reports)
    ("4.1",  "tab:ex1:worked",     "6",  "probes/ex1_v2.json + deployed validator",
     "one frozen state worked through"),
    ("4.2",  "tab:ex1:binding",    "7",  "probes/ex1_v2.json",
     "which constraints bind, states and pairs"),
    ("4.3",  "tab:ex1:subsets",    "8",  "probes/ex1_v2.json",
     "the 162 states split two ways"),
    ("4.4",  "tab:ex1:rungs",      "9",  "experiments/ex1/prompts.py RUNGS",
     "the eight prompt conditions"),
    ("4.5",  "tab:ex1:grid",       "9",  "experiments/ex1/prompts.py RUNGS",
     "the 2x3 crossed design"),
    ("4.6",  "tab:ex1:floors",     "10", "probes/ex1_v2.json + deployed validator",
     "chance floor and width-blind line"),
    ("4.7",  "tab:ex1:design",     "11", "24 cast A run files",
     "legality across the whole design"),
    ("4.8",  "tab:ex1:baseline",   "12", "3 Full Information run files",
     "baseline competence, three populations"),
    ("4.9",  "tab:ex1:composition","13", "9 cast A run files",
     "violations by cause, with legality"),
    ("4.10", "tab:ex1:signatures", "14", "9 cast A run files",
     "three signatures of an unregistered loss"),
    ("4.11", "tab:ex1:castb",      "15", "8 cast B run files + probes/ex1_setb_v1.json",
     "the second object set"),
    ("F.1",  "tab:ex1:terms",      "16", "probes/ex1_v2.json",
     "state counts quoted in the glossary"),
    ("G.1",  "tab:ex1:swap",       "17", "12 cast A run files + mislabel.py",
     "the object-name swap test"),
]

status = dict((label, (st, detail)) for label, st, detail in CHECKS)
rows = []
for number, label, section, source, reports in AUDIT:
    st, detail = status.get(label, ("NOT CHECKED", ""))
    rows.append([number, label, "§" + section, reports, source, st])

show(["Table", "Label", "Built in", "Reports", "Source data", "Against thesis"],
     rows, "Experiment 1 reproducibility audit")

n_match = sum(1 for r in rows if r[-1] == "MATCHES")
print("\n%d of %d tables regenerated and matched against the thesis"
      % (n_match, len(rows)))
failed = [r for r in rows if r[-1] not in ("MATCHES", "SKIPPED")]
assert not failed, "tables not verified: %s" % [r[1] for r in failed]

Experiment 1 reproducibility audit
-----  -------------------  --------  ----------------------------------------  --------------------------------------------  --------------
Table  Label                Built in  Reports                                   Source data                                   Against thesis
-----  -------------------  --------  ----------------------------------------  --------------------------------------------  --------------
4.1    tab:ex1:worked       §6        one frozen state worked through           probes/ex1_v2.json + deployed validator       MATCHES       
4.2    tab:ex1:binding      §7        which constraints bind, states and pairs  probes/ex1_v2.json                            MATCHES       
4.3    tab:ex1:subsets      §8        the 162 states split two ways             probes/ex1_v2.json                            MATCHES       
4.4    tab:ex1:rungs        §9        the eight prompt conditions               experiments/ex1/prompts.py RUNGS       

## What the audit found

Every table above is now generated from the raw data and checked against the
thesis. Four things were true when this notebook was written and are recorded
here rather than silently fixed:

1. **Table 4.7's Legality column had no generator.** The pipeline script
   `analysis/ex1/ex1_results.py` emits an eight-column version of this table
   whose Legality column is **trial level**, plus separate `n` and `Scene`
   columns. The thesis prints a six-column version whose Legality column is the
   **scene majority with a Wilson interval** — the correct unit, and the one the
   chapter reports throughout, but one that was being produced by hand. Section
   11 generates the published form directly. All 24 cells match.

2. **Table 4.8 had no generator at all.** No script in the repository built the
   baseline table; it was assembled by hand from three separate measures.
   Section 12 builds it. All nine cells match.

3. **Table 4.9's Legality column had no generator.** `table_composition` in the
   pipeline emits the violation counts without it. Section 13 builds the column
   and asserts, cell by cell, that it agrees with Table 4.7 rather than merely
   coming close.

4. **Table 4.7's note is wrong in both its range and its cause.** It reads
   "scored on 95 or 96 grasp-binding scenes, the occasional shortfall from a
   single unparseable GPT reply". The true range is **94 to 96** — GPT's No
   Rules and Legal-Arm Control cells are scored on 94 — and the shortfall is
   not caused by unparseable replies. It is caused by GPT **declining on every
   repeat of a scene**: a decline is not a proposal, so a scene with no
   proposals has no majority to take and leaves the denominator. The one
   genuinely unparseable reply in cast A is Gemini's, in No Width + Swapped,
   and it costs a trial rather than a scene. Section 11 establishes this from
   the data and the generated note says it.

5. **Table 4.8's column header understates one denominator.** The header prints
   `n=126` over the all-picking column for all three models. GPT is scored on
   125 scenes, because one scene has no majority-eligible repeat — the same
   reason the design table's note says "95 or 96". The printed cell values are
   correct; only the header is one state optimistic. The generated LaTeX in
   section 12 puts the true per-model denominators in the table note.

Two further notes for anyone re-running this:

- **Table 4.7 prints two units in one table.** Legality is the scene majority;
  the negative control is trial level. Section 11 computes both and the
  diagnostic cell shows they support the same reading. If the caption is ever
  revised, this is the fact it needs to state.
- **`docs/TABLE_PROVENANCE.md` contradicts the code on declines.** It says a
  decline "is scored rather than dropped"; the pipeline and this notebook both
  drop it, which is what makes the trial denominators vary between cells. The
  thesis glossary has it right — "Declines are not in the denominator" — so the
  documentation is what needs correcting, not the numbers.